![](../../assets/header-logo.png)

## *AUTOLABELING TURN INDICATORS WITH FOUNDATION MODELS*

**Automated and Connected Driving Challenges - Research Project - *09/2026***

**Research Topic *17: Autolabeling Turn Indicators with Foundation Models***

| Author | Mail |
| --- | --- |
| *Benny Samraj Linda* | *benny.samraj-linda@rwth-aachen.de* |
| *Goutham Tulluri* | *goutham.tulluri@rwth-aachen.de* |
| *Shashank Radhakrishnan* | *shashank.radhakrishnan@rwth-aachen.de* |

## Dependencies & Setup

Python 3.10 is required to use hugging face's latest algorithms.

After running this cell, **refresh the browser tab**. In the top right corner of your notebook, change your Kernel to **Python 3.10 (SAM 3.1)**.(Expected time ~3 minutes)

In [ ]:
%%bash

# 1. System Dependencies
echo "jovyan" | sudo -S apt-get update && sudo -S apt-get install -y ros-noetic-sensor-msgs libgl1

# 2. Create Conda Environment
conda create -n sam_env python=3.10 -y

# 3. Install Dependencies from 'requirements.txt'
conda run -n sam_env pip install ipykernel

conda run -n sam_env pip install -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu124

conda run -n sam_env pip install git+https://github.com/ByteDance-Seed/depth-anything-3.git@3d835ec1a5802d64a8b8b15f817a1ab54809bfe4

# 4. Register Kernel
conda run -n sam_env python -m ipykernel install --user --name sam_env --display-name "Python 3.10 (SAM 3.1)"

Import the Python packages required for the code cells in this report.

In [ ]:
import sys, os, io, ast, cv2, time, json, rosbag, psutil, torch, math
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse, Rectangle
from matplotlib.gridspec import GridSpec
import matplotlib.patheffects as pe
from contextlib import redirect_stdout
from scipy.interpolate import griddata
from scipy.signal import find_peaks
import numpy as np
import pandas as pd
from IPython.display import display, Image
from tqdm.std import tqdm
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from depth_anything_3.api import DepthAnything3
from transformers import Sam3Processor, Sam3Model

To load Hugging Face models, One must have authenticated tokens to download the weights.

   - Go to Hugging Face SAM 3.1 and click Agree to accept Meta's license.
   - Go to your Hugging Face Settings -> Access Tokens and generate a Read token. Keep it handy for the execution script.

A sample token is used ("Sample_Token").

In [ ]:
my_hf_token = "Sample_Token" # Sample token

# Provide Hugging Face token here
os.environ["HF_TOKEN"] = my_hf_token

Minimum 10 gb of gpu required to run both DA3-Giant (major usage) and SAM 3.1 Foundation models.

In [ ]:
!nvidia-smi --query-gpu=memory.total,memory.used,memory.free --format=csv

Pull required functions.

In [ ]:
from functions.transform import generate_camera_transform, project_lidar_to_2d
from functions.common import read_timesync_file, load_calibration_file
from functions.annotations import convert_labels_from_tracks_wise_to_timestamp_wise

### Downloads

Dataset 'data_night_sample' should be copied to the folder 'data' from 'acdc-rp-samraj-linda'.

## 1. Abstract

Accurately detecting turn indicator states can provide vital information for autonomous driving systems to anticipate and plan according to the intent of surrounding vehicles for safe motion planning. However, the task of manual annotation of turn indicator states, in autonomous driving datasets is both costly and time-consuming. In this report, we explore an automated methodology to extract and label turn indicator states using foundation models. Our pipeline utilizes the DrivIng [[1]](#ref) dataset which features 18 kms of continuous tracked vehicle data captured using a 6xRGB cameras capturing and LiDAR, both capturing 360 degrees across day, and night scenarios. Furthermore, we segment indicators using Segment Anything Model (SAM 3.1) [[2]](#ref) and associate these 2D masks into the 3D vehicle frame by estimating metric depth using a fusion of Depth Anything 3 (DA3) [[3]](#ref) and LiDAR point cloud. Subsequently, we then estimate the indicator state by employing colour and frequency-based filtering. In brief this filter first extracts the amber pixels and then applies a rolling median combined with autocorrelation function to isolates - tag the periodic blinking of the indicators. This multi-strategy is essentially required to overcome challenges such as but not limited to overexposure and random brake light activations. Finally, a manual sample evaluation of 85 Day and 50 Night frames was done, to test the effectiveness of the pipeline and the indicator state estimation algorithm.

## 2. Introduction

The ability to perceive and interpret indicator signals provides information on the target vehicle’s intent; this enables an autonomous system to transition from a reactive mode,to a proactive one. It would help the system to anticipate lateral movement well in advance even before they physically begin. This early detection capability can directly lead to increased awareness during emergency braking, smoother trajectory planning, and safer interaction with human-driven vehicles at intersections and on highways.

Despite their importance, turn indicator states are rarely included as labeled attributes in public autonomous driving datasets. This gap exists because manual frame-by-frame annotation is labor and cost intensive: a human annotator must inspect each camera frame - identify every visible indicator light, and determine its indicator state across thousands of frames. State-of-the-art datasets such as nuScenes [[4]](#ref), Waymo Open Dataset [[5]](#ref), KITTI [[6]](#ref) and DrivIng [[1]](#ref) provide 3D object bounding boxes and track IDs but omit indicator state annotation entirely. 

Among these datasets we choose DrivIng [[1]](#ref) created by Computer Vision for Intelligent Mobility Systems (CVIMS) group at TH Ingolstadt. It contains 18 kms of continuously labelled driving data, in diverse lighting conditions across day and night. This makes it valuable for testing perception robustness under varying illumination and overexposure. It utilises 1 Mid mounted LiDAR along with 6 synchronized RGB cameras (Front Left, Front Right, Left, Right, Back Left, Back Right) providing surround-view coverage. The vehicles and objects are annotated with 3D bounding boxes and are tracked across frames using specific 'track IDs'. This temporal continuity allows us to analyze frequency based attributes like indicators. Additionally, the dataset has a capture rate of 10 frames per second, ensuring that indicator blinking is never missed – as per ECE [[7-8]](#ref) standards the indicator frequency must be within 1-2Hz and each blink must be in ON state for at least 250ms with 50% duty cycle [[7-8]](#ref).

Our work fills the gap of indicator annotation process by utilizing a fully automated pipeline built on the following four stages:

1.  Segmentation of head- and tail-light clusters: For this purpose we utilize a foundation model named Segment Anything Model 3.1 (SAM3), which isolates candidate indicator mask regions from RGB frames just by using simple text prompts.
2. Depth Estimation: Depth metric is required to match the SAM3 detections to the corresponding trackID’s, this is achieved by fusing the relative depth maps generated from the second foundation model used called Depth Anything 3 (DA3) with sparse LiDAR point cloud data from DrIving. DA3 was particularly used to overcome the sparseness of the LiDAR point cloud
3. Geometric Association and Classification: To project mask centroids into 3D space and map them to the local coordinate frame of each vehicle to rightly tag the indicators as left or right.
4. Indicator State Estimation: Finally, analyse and estimate the indicator states across multiple frames to classify them as ON or OFF.

## 3. Related Work

The development of perception algorithms for autonomous driving relies heavily on the availability of diverse datasets. While earlier datasets provided standard camera and LiDAR modalities, recent trends have shifted toward synchronization between sensors and how true they represent the real world. Most notably, Rößle et al. introduced DrivIng [[1]](#ref), a large-scale multimodal driving dataset distinguished by its complete digital twin integration provides synchronized data across its sensors. The DrivIng dataset serves as an ideal foundational testbed for our work, enabling the precise spatial and temporal alignment required for multi-sensor fusion and vehicle state annotation.

The capabilities of computer vision have expanded significantly by the introduction of foundation models capable of performing accurate segmentation and depth estimation from images. Hence, it allows us to isolate specific features of a vehicle—such as headlights or taillights— which would otherwise require bespoke training to obtain specialized neural networks. The Meta FAIR’s release of Segment Anything 3 (SAM 3) [[2]](#ref) has helped us bypass this step altogether. SAM3 makes it intuitive by accepting flexible text prompts enabling easy extraction of indicators. Moreover, to adapt the masks against their corresponding track ID’s, requires a dense depth map to augment the sparse LiDAR depth data.

Extracting 3D spatial information from 2D images has seen rapid progress with the prolific use of vision based systems. Lin et al. recently proposed Depth Anything 3 (DA3) [[3]](#ref), a foundational model that's capable of depth estimation from 2D images. To deploy such models for real-world applications, efforts such as by Beemelmanns [[9]](#ref) has successfully utilized metric variants of DA3 via ROS2 and TensorRT for real-time point cloud generation.

Furthermore, the foundational techniques utilized in this study for object association and sensor fusion draw direct inspiration from the ‘Automated and Connected Driving Challenges” MOOC offered by RWTH Aachen University [[10]](#ref). This course provides a concise, accessible, and structure for Object Association using- and Sensor Data Fusion between- LiDAR and camera frames.

Finally, previous approaches to turn signal detection have transitioned from rigid algorithm to supervised deep learning. Early methods, such as the lightweight system proposed by Casares et al. [[11]](#ref), relied on color-based symmetry constraints paired with Kalman filtering to track rear lights, classifying turning and braking states by monitoring luminance variations over time, but this technique was found to be lacking in overexposed lighting conditions. To address this sensitivity of manually defined visual thresholds, subsequent research introduced spatio-temporal deep learning techniques. Notably, Hsu et al. [[12]](#ref) developed a CNN-LSTM architecture that isolates dynamic turn signals by computing image differences between frames, and learning the temporal dependencies of various rear signal states. While these methods established the necessity of temporal sequence analysis for accurate signal classification, they rely heavily on either rigid heuristic tuning or task-specific supervised training datasets. The reliability required for labelling is uncontrolled in the latter method, while in the former overexposed images would require dynamic threshold setting. In contrast, our approach completely bypasses task-specific model training by utilizing, but utilises an algorithm that uses Lab* colour space for its easy tunability and a frequency based strategy to rightly estimate the indicator state.


## 4. Methodology

The proposed methodology presents a multi-stage pipeline designed to extract, associate, and classify vehicle indicators from the DrivIng dataset. To handle the complexities of overlapping vehicles and ambiguous lighting conditions, the pipeline is structured into the following stages:
   
   - 4.1 Dataset & Preprocessing
   
   - 4.2 Dense Depth Map Generation

   - 4.3 Turn Indicator Segmentation
   
   - 4.4 Indicator Association & Classification
   
   - 4.5 Mask Amber Computation
   
   - 4.6 Main Pipeline
   
   - 4.7 Blinking State Estimation
   
   - 4.8 Multi-Camera Data Fusion

### 4.1 Dataset and Preprocessing
As our pipeline utilizes DrivIng, a multimodal autonomous driving dataset featuring six RGB cameras, LiDAR, and a fully geo-referenced digital twin. It provides over 1.2 million 3D bounding box annotations at 10 Hz across diverse urban and highway environments under varying lighting conditions. To project these 3D annotations into 2D camera space, we leverage the matrices and base utility functions from the official DrivIng repository [[1]](https://github.com/cvims/DrivIng).
  

Dataset Structure:
<pre>
  ├── back_left_camera
  ├── back_right_camera
  ├── front_left_camera
  ├── front_right_camera
  ├── left_camera
  ├── middle_lidar
  ├── right_camera
  ├── sweeps
  ├── vehicle_state
  ├── annotations.json
  ├── calibration.json
  └── timesync_info.csv
</pre>

In [ ]:
# Path to relevant dataset
seq_dir = "data/data_night_sample"

# 1. Load time sync data for 6 cameras and middle lidar
sync_table = read_timesync_file(os.path.join(seq_dir, 'timesync_info.csv'))

# 2. Load calibration files for cameras and lidar
calib_data = load_calibration_file(os.path.join(seq_dir, 'calibration.json'))

# 3. Load and pivot annotations natively
with open(os.path.join(seq_dir, 'annotations.json'), 'r') as f:
    raw_annots = json.load(f)
    
annots_time_wise = convert_labels_from_tracks_wise_to_timestamp_wise(raw_annots)

# Inspect the first object in the first frame to see the fields 
all_timestamps = list(annots_time_wise.keys())
target_ts = all_timestamps[0]
first_obj = annots_time_wise[target_ts][0]
print(f"Object keys available: {list(first_obj.keys())}")
print(first_obj)

#### 4.1.1 Geometric Projection

The 3D spatial coordinates are transformed from **the LiDAR coordinate system to the vehicle frame, and subsequently to the target camera frame**, using their respective extrinsic matrices. The points are then projected onto the 2D image plane using the specific camera’s intrinsic matrix and distortion coefficients. 

While we leverage the base projection matrices and utility functions provided by the official DrivIng dataset repository [[1]](https://github.com/cvims/DrivIng), we further refine this logic to systematically **group the resulting 8-point 2D coordinates** along the longitudinal axis. This allows us to extract mathematically precise front and rear vehicle faces, which serve as the geometric foundation for our indicator classification.

<div align="center">
  <figure>
    <img src="images/8-point-2d-image.png" width="600" alt="2D Bounding Box">
    <figcaption><b>Fig. 1:</b> 2D Bounding Box representation showing the 8-point 2D projection on the image plane.</figcaption>
  </figure>
</div>

#### 4.1.2 Temporal Synchronization

Inter-sensor temporal alignment was achieved by using the time_sync_info.csv file from the DrivIng dataset [[1]](https://github.com/cvims/DrivIng). Executing the dataset's native synchronization function, we successfully mapped individual camera temporal indices to a unified global master (LiDAR) timestamp, standardizing the temporal domain across our processed dataset.

In [ ]:
camera_names = [
    'back_left_camera', 'left_camera', 'front_left_camera',
    'front_right_camera', 'right_camera', 'back_right_camera'
]

# Dictionary to store all our generated camera DataFrames
camera_dataframes = {}

print("\nProcessing 2D Camera Projections...")

for cam in camera_names:
    df_cam = generate_camera_transform(
        target_camera=cam, 
        sync_table=sync_table, 
        calib_data=calib_data, 
        annots_time_wise=annots_time_wise)
    
    camera_dataframes[cam] = df_cam
    print(f" ~ {cam}: {len(df_cam)} 2D bounding boxes extracted.")

In [ ]:
pd.set_option('display.max_columns', None) 
display(camera_dataframes['left_camera'].head(10))

### 4.2 Depth Estimation and Fusion

To overcome the inherent sparsity and false object reflections of LiDAR, we integrate **Depth Anything 3 (DA3)** [[3]](#ref). DA3 is a foundational model for monocular depth estimation, designed to generate dense, *continuous depth maps* directly from 2D camera images. 

While the metric variants of DA3 offer absolute depth outputs, our empirical evaluations—conducted using the implementation provided in [[4]]((#ref)) revealed notable discrepancies when benchmarked against ground-truth LiDAR measurements. But even very small inaccuracies like 5m leads to inaccurate indicator associations in our case. To resolve this, we propose a fusion strategy that uses the DA3 relative depth model in combinnation with the raw LiDAR point cloud to compute the absolute depth. We first superimpose the points using x, y coordinates between DA3 & LiDAR. Post mapping, we then scale the relative depth of DA3 using LiDAR depth.

| Model | Encoder | Params | Output Type |
| :--- | :--- | :--- | :--- |
| DA3-Small | ViT-S | ~25 M | Relative Depth |
| DA3-Base | ViT-B | ~120 M | Relative Depth |
| DA3-Large | ViT-L | ~350 M | Relative Depth |
| DA3-Giant | ViT-G | ~1.15 B | Relative Depth |
| DA3-Metric-Large | ViT-L | ~350 M | Absolute Depth |
| DA3-Nested-Giant-Large | ViT-G | ~1.40 B | Absolute Depth |

To generate the dense continuous depth profile, our pipeline uses the "Giant" variant of Depth Anything 3 (DA3-Giant), which gives the best accuracy in relative depth segment.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Initializing Depth Anything V3 Giant on: {device}...")

# Pulling the heavy ViT-Large architecture
model_DA3 = DepthAnything3.from_pretrained("depth-anything/da3-giant")
model_DA3 = model_DA3.to(device).eval()

print("DA3-Giant loaded successfully!")

**Function Overview: `get_dense_metric_depth`**
This function generates an accurate, dense metric depth map by fusing relative depth predictions from DA3 with LiDAR ground-truth data.

*Inputs:*

* **`image_np:`** The 2D input image array.
* **`lidar_u, lidar_v:`** x, y coordinates of LiDAR.
* **`lidar_cam_z:`** z coordinates of LiDAR.
* **`processor, model, device:`** DA3 model and hardware target.

*Core Mechanism:*

* **`Relative Depth Estimation:`** It processes the 2D input image through DA3 to generate a dense, relative depth map $D_{rel}(u, v)$.
* **`Scale Factor Calculation:`** It aligns valid - sparse LiDAR points $Z_{LiDAR}(u_i, v_i)$ with their corresponding pixels on the relative depth map to calculate exact, localized scale factors $S_i$. Since the model output is directly proportional to depth, the scale is calculated via division:
  $$ S_i = \frac{Z_{LiDAR}(u_i, v_i)}{\max(D_{rel}(u_i, v_i), 10^{-6})} $$
  
* **`Zonal Scaling (Spatial Interpolation):`** Rather than applying a single global scaling factor, it uses 2D spatial interpolation (`scipy.interpolate.griddata`) to smoothly blend these local scale factors into a continuous 2D scale map $S(u, v)$. The final metric depth is then computed by multiplying this interpolated scale map with the original relative depth:
  $$ Z_{metric}(u, v) = S(u, v) \times D_{rel}(u, v) $$

*Outputs:*

* **`metric_depth:`** A 2D numpy array (size of image) where every pixel contains its absolute, physically scaled depth in meters.

<div align="center">
  <figure>
    <img src="images/depth-map.png" width="1400" alt="Depth Map">
    <figcaption><b>Fig. 2:</b> Comparison between LiDAR pcd and metric depth map.</figcaption>
  </figure>
</div>

In [ ]:
def get_dense_metric_depth(image_np, lidar_u, lidar_v, lidar_cam_z, processor, model, device):
    """
    Generates a dense, metric depth map by fusing relative DA3 depth predictions 
    with sparse LiDAR ground-truth data.

    Args:
        image_np (np.ndarray): The 2D RGB input image array.
        lidar_u (np.ndarray): 1D array of horizontal (x) image coordinates for LiDAR hits.
        lidar_v (np.ndarray): 1D array of vertical (y) image coordinates for LiDAR hits.
        lidar_cam_z (np.ndarray): 1D array of metric depth (z) values for the LiDAR hits.
        processor (object or None): The HuggingFace processor. If None, assumes Depth Anything V3.
        model (object): The instantiated depth estimation model (HF or custom).
        device (str/torch.device): The hardware acceleration target (e.g., 'cuda' or 'cpu').

    Returns:
        np.ndarray: A 2D dense depth map (in meters) matching the original image dimensions.
    """
    H_img, W_img = image_np.shape[:2]
    
    # 1. Generate Relative Depth Map via DA3 Foundation Model
    if processor is None:
        # Depth Anything V3 Path (Custom Model)
        with torch.no_grad():
            
            # Suppress noisy [INFO] prints from the model internals during inference
            with open(os.devnull, 'w') as f, redirect_stdout(f):
                if hasattr(model, 'infer_image'):
                    raw_depth = model.infer_image(image_np) # DA v2/v3 standard API
                elif hasattr(model, 'inference'):
                    pred = model.inference([image_np]) # Fallback for some wrapper APIs
                    raw_depth = pred.depth[0] if hasattr(pred, 'depth') else pred[0]
                else:
                    # Fallback for raw torch modules without wrappers
                    raw_depth = model(torch.from_numpy(image_np).permute(2, 0, 1).unsqueeze(0).float().to(device))
                    raw_depth = raw_depth.squeeze().cpu().numpy()
                
        # Resize DA3 prediction to match original image shape if necessary
        if raw_depth.shape[:2] != (H_img, W_img):
            disparity = cv2.resize(raw_depth, (W_img, H_img), interpolation=cv2.INTER_CUBIC)
        else:
            disparity = raw_depth
            
    else:
        # HuggingFace Transformers Path
        pil_image = Image.fromarray(image_np) if not isinstance(image_np, Image.Image) else image_np
        inputs = processor(images=pil_image, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            predicted_depth = outputs.predicted_depth
    
        # Interpolate HuggingFace output back to native resolution
        prediction = torch.nn.functional.interpolate(
            predicted_depth.unsqueeze(1),
            size=(H_img, W_img),
            mode="bicubic",
            align_corners=False,
        )
        disparity = prediction.squeeze().cpu().numpy()
    
    # 2. Filter Valid LiDAR Points
    H, W = disparity.shape
    
    # Keep only LiDAR points that are in front of the camera (z > 0)
    valid_mask = (lidar_u >= 0) & (lidar_u < W) & (lidar_v >= 0) & (lidar_v < H) & (lidar_cam_z > 0)
    
    u = lidar_u[valid_mask]
    v = lidar_v[valid_mask]
    z = lidar_cam_z[valid_mask]
    
    # Failsafe: If no valid LiDAR points exist to anchor the scale, return a static default depth
    if len(z) < 10:
        return np.ones((H, W)) * 10.0 
        
    # Extract DA3 predicted depth exactly where LiDAR hits exist
    disp_sampled = disparity[v, u]
    
    # Avoid zero division when calculating scale factors
    valid_disp = disp_sampled > 1e-6
    if not np.any(valid_disp):
        return np.ones((H, W)) * 10.0
        
    disp_sampled = disp_sampled[valid_disp]
    z = z[valid_disp]
    u = u[valid_disp]
    v = v[valid_disp]
    
    # 3. Compute and Interpolate Metric Scale
    # The DA3 outputs relative depth directly. 
    # Therefore: Z_metric = Scale * DA3_Depth -> Scale = Z_metric / DA3_Depth
    sparse_scales = z / np.clip(disp_sampled, 1e-6, None)
    global_median_scale = np.median(sparse_scales)
    
    # Create geometric coordinate grids to perform 2D spatial interpolation
    grid_x, grid_y = np.meshgrid(np.arange(W), np.arange(H))
    points = np.vstack((u, v)).T
    
    # Interpolate the sparse scale factors across the entire 2D image.
    # Between LiDAR hit points: Linear method creates a smooth Delaunay triangulation.
    # Outside LiDAR hit points (e.g. the sky): global median scale is used.
    dense_scale_map = griddata(
        points, sparse_scales, (grid_x, grid_y), 
        method='linear', fill_value=global_median_scale
    )
    
    # Fallback to nearest-neighbor if linear triangulation fails (e.g., if all points are perfectly collinear)
    if np.any(np.isnan(dense_scale_map)):
        dense_scale_map = griddata(
            points, sparse_scales, (grid_x, grid_y), 
            method='nearest', fill_value=global_median_scale
        )
    
    # 4. Depth Fusion
    # Project the localized 2D scale map over the raw DA3 disparity to generate accurate dense metric depth
    metric_depth = dense_scale_map * disparity
    
    return metric_depth

### 4.3 Turn Indicator Segmentation
To extract the precise boundaries of turn indicators, we deploy the **Segment Anything Model 3.1 (SAM 3.1)** [[2]](https://huggingface.co/facebook/sam3).

Specifically, our pipeline implements the base facebook/sam3 weights, which utilizes a Vision Transformer (ViT) architecture divided into three core engines: a heavy image encoder, a flexible prompt encoder, and a lightweight mask decoder. By loading the uncompressed base model, we ensure maximum feature extraction for small, ambiguous targets. The ViT image encoder processes the 2D camera frame into a dense spatial embedding just once per frame. Subsequently, our text prompt (e.g., "car taillight") are processed by the prompt encoder and fed into the mask decoder alongside the image embedding. This architecture allows the model—running in half-precision (float16) on CUDA hardware—to rapidly predict and shrink-wrap pixel-perfect boundaries for multiple turn indicators in real-time, without suffering the computational penalty of re-encoding the high-resolution driving image for every new target prompt.

In [ ]:
import transformers
transformers.logging.disable_progress_bar()

# Define the hardware device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the Processor
processor = Sam3Processor.from_pretrained("facebook/sam3")

# Load the massive Foundation Model into your GPU
model = Sam3Model.from_pretrained("facebook/sam3", torch_dtype=torch.float16).to(device)

print("Setup complete! Processor and Model are loaded into memory.")

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())

print(f"Total Parameters used in SAM3.1 model: {total_params:,}")

**Function Overview: `generate_sam3_masks`**
This function uses the Segment Anything Model (SAM 3) architecture to generate precise segmentation masks based on text prompts, returning the results directly as PyTorch GPU tensors to minimise time.

*Inputs:*

* **`image_path:`** The source image (file path or loaded object).
* **`text_prompt:`** The target to mask (e.g., "car taillight").
* **`exclude_prompt (Optional):`** A secondary prompt used to create a negative boundary (e.g., "red car chassis").
* **`processor, model, device:`** The SAM 3 model and hardware target.
* **`conf_threshold, mask_threshold:`** Numerical limits to filter out low-confidence predictions.

*Core Mechanism:*

* **`Mixed-Precision Inference:`** It processes the inputs using hardware-accelerated half-precision (torch.float16) to optimize memory and inference speed.
* **`Semantic Targeting:`** It extracts initial segmentation masks and confidence scores based on the primary text prompt.
* **`Negative Prompting (Exclusion Logic):`** If an exclude prompt is provided, it generates a secondary set of masks to create a boolean "exclude zone". It mathematically subtracts this zone from the target masks.

*Outputs:*

* **`target_masks:`** A boolean PyTorch GPU tensor containing the final, valid 2D segmentation masks.
* **`target_scores:`** A 1D PyTorch GPU tensor containing the confidence score for each preserved mask.

In [ ]:
def generate_sam3_masks(image_path, text_prompt, processor, model, device, exclude_prompt=None, conf_threshold=0.15, mask_threshold=0.5):
    """
    Generates semantic masks using SAM3 and returns them strictly as PyTorch GPU Tensors.
    
    Args:
        image_path (str or PIL.Image): Path to the image or the loaded PIL Image.
        text_prompt (str): The primary text prompt for the target object (e.g., 'car taillight').
        processor: The SAM3 processor instance.
        model: The SAM3 model instance.
        device: The PyTorch device (e.g., 'cuda' or 'cpu').
        exclude_prompt (str, optional): A text prompt for regions to actively subtract/ignore.
        conf_threshold (float): Confidence threshold for instance detection.
        mask_threshold (float): Threshold for binarizing the segmentation masks.
        
    Returns:
        tuple: (target_masks, target_scores) both as raw GPU PyTorch Tensors.
    """
    if isinstance(image_path, str):
        image = Image.open(image_path).convert("RGB")
    else:
        image = image_path.convert("RGB")

    orig_w, orig_h = image.size

    prompts = [text_prompt]
    if exclude_prompt is not None:
        prompts.append(exclude_prompt)

    inputs = processor(images=image, text=prompts, return_tensors="pt")
    inputs = {k: v.to(device=device, dtype=torch.float16 if v.dtype == torch.float32 else None) for k, v in inputs.items()}

    device_type = "cuda" if "cuda" in str(device) else "cpu"
    with torch.inference_mode(), torch.amp.autocast(device_type=device_type, dtype=torch.float16):
        outputs = model(**inputs)

    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=conf_threshold,
        mask_threshold=mask_threshold,
        target_sizes=[(orig_h, orig_w)] * len(prompts)
    )

    # Keep masks and scores on the GPU to minimize host transfer overhead
    target_masks = results[0]["masks"]
    target_scores = results[0]["scores"]

    if len(target_masks) == 0:
        return target_masks, target_scores

    # Filter out regions intersecting with the exclusion prompt
    if exclude_prompt is not None and len(results) > 1:
        exclude_masks = results[1]["masks"]
        if len(exclude_masks) > 0:
            forbidden_zone = torch.any(exclude_masks, dim=0)
            target_masks = target_masks & ~forbidden_zone
            valid_mask = target_masks.sum(dim=(1, 2)) > 10
            target_masks = target_masks[valid_mask]
            target_scores = target_scores[valid_mask]

    return target_masks, target_scores

### 4.4 Indicator Association and Classification 

#### 4.4.1 Mask Association

After mask generation, the pipeline must accurately attribute each indicator to a specific vehicle represented in dataset by **track id**. 

To handle different scenarios we implemented a two-tier association logic:
 
Initially masks are passed through a 2D filter (3D LiDAR bounding box flattened and overlayed onto the 2D image), 
- If a single candidate is identified, it is checked with 3D LiDAR Bounding Box filter - this decides if we can use depth for classification or not.
- If multiple candidates are identified - 3D LiDAR Bounding Box filter is used as the resolution tool. 

In State 0 and State 1, masks are assoicated to a track id, while State 2 means mask is rejected.

<div align="center">
  <figure>
    <img src="images/association-logic.png" width="1000" alt="Association Logic">
    <figcaption><b>Fig. 3:</b> Logical flow of 2D-3D associations.</figcaption>
  </figure>
</div>

##### 4.4.11 Mask Association in 2D
A filtering mechanism utilizes the precomputed 2D vehicle bounding boxes available in the frame.

* **Candidate Selection:** Even if a single pixel of a generated SAM 3 mask falls within the vehicle's 4-point 2D bounding box (4 extremes from 8 point 2D coordinate bounding box), that indicator is registered as a candidate.
* **Unambiguous Association:** If the filter yields exactly one candidate vehicle (one bounding box for a mask), the target is deemed unambiguous. The mask is immediately declared "Associated" and assigned to that specific vehicle's Track ID

<div align="center">
    <img src="images/2d-unambiguous.png" width="400">&nbsp;&nbsp;&nbsp;&nbsp;<img src="images/2d-ambiguous.png" width="543">
    <br><br>
    <i><b>Fig. 4:</b> Comparison of 2D bounding box association scenarios. <b>(Left)</b> Unambiguous-Single Candidate <b>(Right)</b> Ambiguous-Multiple Candidates</i>
</div>

A mask is left to 3D association to decide incase of ambiguous association:

##### 4.4.12 Mask Association in 3D
* **Unambiguous 2D Associations:** In case of successful 3D association, the depth value is recorded. But, if the depth contradicts 2D association, we retain the 2D association and discard the depth value.
* **Ambiguous 2D Associations:** When a mask intersects multiple overlapping 2D bounding boxes, we use 3D association logic to find a bounding box match. If there is no match in 3D association, the mask gets rejected.

First, the spatial offset between the True Depth (3D centroid) of the detected mask and the 3D geometric center of the vehicle is calculated in the global coordinate frame to find the deltas ($\Delta x$, $\Delta y$).

A 2D rotation matrix, parameterized by the vehicle's 3D yaw angle, is then applied to project this offset vector into the vehicle's localized coordinate system:

$$
\begin{bmatrix} l_x \\ l_y \end{bmatrix} = \begin{bmatrix} \cos(\text{yaw}) & \sin(\text{yaw}) \\ -\sin(\text{yaw}) & \cos(\text{yaw}) \end{bmatrix} \begin{bmatrix} \Delta x \\ \Delta y \end{bmatrix}
$$

<div align="center">
  <figure>
    <img src="images/3d-Association.png" width="600" alt="3D Bounding Box">
    <figcaption><b>Fig. 5:</b> Bird's eye view in ego vehicle frame</figcaption>
  </figure>
</div>

In this local frame, the longitudinal axis ($l_x$) aligns with the vehicle's heading, while the lateral axis ($l_y$) represents the orthogonal side-to-side orientation.

The association is geometrically confirmed if the localized coordinates satisfy the following boundaries:

$$
\vert{}l_x\vert{} \leq \frac{W}{2} \quad \text{and} \quad \vert{}l_y\vert{} \leq \frac{L}{2}
$$

#### 4.4.2 Mask Classification

Once association is completed, we get a list of masks for a track id. For simplicity, top 2 confidence scored (SAM3 Confidence) masks are selected and classified into left and right.

Comparing two masks is not the correct idea because of occluded indicators, where only one mask is available per track id. Another problem associated with this approach is when vehicle is at 180 degrees with respect to camera, where both indicators visible are left or right.

Hence each mask must be individually classified as left or right based on geometry.

$$
State-0 : \text{2D Classification}
$$

$$
State-1 : \text{3D Classification}
$$

##### 4.4.21 2D Classification Logic

Distance to right and left edges in 2D bounding box are calculated and minimum Euclidean distance to an edge is used as criteria for classification. Front indicators are classified using front bounding box and rear indicator using rear bounding box.

<div align="center">
  <figure>
    <img src="images/2d-classification.png" width="700" alt="2D Classification">
    <figcaption><b>Fig. 6:</b> 2D classification example for front-left indicator</figcaption>
  </figure>
</div>


##### 4.4.22 3D Classification Logic

Mathematically, the calculation of the lateral coordinate ($l_y$) is equivalent to the 2D cross product between the vehicle's heading vector and the mask's relative position vector. By evaluating the sign of the localized lateral coordinate, the system deterministically resolves spatial ambiguity, assigning the indicator to the left or right side of the vehicle regardless of its global orientation or trajectory.

$$
l_y > 0 \implies \text{Left Indicator}
$$

$$
l_y < 0 \implies \text{Right Indicator}
$$

<div align="center">
  <figure>
    <img src="images/3d-classification.png" width="700" alt="2D Classification">
    <figcaption><b>Fig. 7:</b> 3D classification representation for front-left indicator</figcaption>
  </figure>
</div>

**Function Overview: `associate_masks_to_boxes`**
This function fuses 2D bounding boxes and 3D depth data along with generated segmentation masks to the vehicles in the frame.

*Inputs:*

* **`df_2d_boxes:`** A Pandas DataFrame containing vehicle tracking data (2D bounding boxes, 3D centroids, yaw, depth, and 2D corner projections...).
* **`masks_tensor, scores_tensor, is_rear_flags:`** GPU tensors representing the segmented masks, their confidence scores, and categorical flags indicating whether the mask corresponds to the front or rear of a vehicle.
* **`amber_stats, color_stats, centroids_3d:`** Lists containing the previously computed mask metrics (pixel area, HSV color statistics, and physical 3D (X,Y,Z) coordinates in meters).
* **`min_mask_area, num_generated, num_valid:`** Tunable parameters for filtering out noise and logging execution statistics.

*Outputs:*

* **`updated_rows:`** A structurally enhanced version of the input df_2d_boxes. It appends the successfully associated mask metrics to left and right indicator columns with features. It also includes an associated_masks column detailing every mask successfully claimed by that specific vehicle.

In [ ]:
def associate_masks_to_boxes(
    df_2d_boxes, 
    masks_tensor,
    scores_tensor,
    is_rear_flags,      
    pixel_counts, 
    centroids_3d, 
    min_mask_area=20, 
    num_generated=0,
    num_valid=0
):
    """
    Associates 2D segmentation masks with 3D/2D vehicle bounding boxes.
    
    This function uses a hybrid approach, matching masks via 2D bounding polygon
    intersection and resolving ties or ambiguities using 3D spatial distancing.
    
    Args:
        df_2d_boxes (pd.DataFrame): Frame-level bounding box and 3D pose data for vehicles.
        masks_tensor (torch.Tensor): PyTorch tensor containing the generated masks.
        scores_tensor (torch.Tensor): PyTorch tensor containing confidence scores for masks.
        is_rear_flags (list of bool): Flags indicating if each mask targets the rear of a vehicle.
        pixel_counts (list of int): Total non-black pixels per mask (cheap, no color overhead).
        centroids_3d (list): The computed 3D centroids for each mask (if depth is available).
        min_mask_area (int): Minimum pixel area for a mask to be considered valid.
        num_generated (int): Total number of masks initially generated by the model.
        num_valid (int): Number of masks that passed the minimum area threshold.
        
    Returns:
        pd.DataFrame: A new dataframe containing updated indicator columns with the matched masks.
    """
    
    # Set up image dimensions and device
    if len(masks_tensor) > 0:
        device = masks_tensor.device
        img_h, img_w = masks_tensor.shape[1], masks_tensor.shape[2]
    else:
        device = "cpu"
        img_h, img_w = 1080, 1920
        
    track_wins = {row['track_id']: [] for _, row in df_2d_boxes.iterrows()}

    # Precompute Car Face Masks and Push to GPU
    cars = []
    for _, row in df_2d_boxes.iterrows():
        cam_name = row['camera']
        track_id = int(row['track_id'])
        yaw_val = row.get('yaw_3d', None)
            
        w, h = row.get('box_width', 0), row.get('box_height', 0)
        box_area = w * h
        car = {
            'track_id': track_id,
            'row': row,
            'cam_name': cam_name,
            'yaw_3d': yaw_val,
            'box_area': box_area
        }
        
        # Prepare CPU masks for intersection logic
        cx, cy = row['center_x'], row['center_y']
        w, h = row.get('box_width', 0), row.get('box_height', 0)
        xmin, ymin = int(cx - (w / 2)), int(cy - (h / 2))
        xmax, ymax = int(cx + (w / 2)), int(cy + (h / 2))
        
        for f_type in ['front', 'rear']:
            c_str = row.get(f"{f_type}_corners_2d", None)
            valid = False
            mask_np = np.zeros((img_h, img_w), dtype=np.uint8)
            
            if type(c_str) != float and c_str is not None:
                try:
                    import ast
                    pts = np.array(ast.literal_eval(c_str)) if isinstance(c_str, str) else np.array(c_str)
                    if len(pts) > 0:
                        hull = cv2.convexHull(pts.astype(np.int32))
                        cv2.fillPoly(mask_np, [hull], 1)
                        valid = True
                except:
                    pass
            
            # Fallback to standard rectangular bounding box if corner points fail
            if not valid:
                cv2.rectangle(mask_np, (max(0, xmin), max(0, ymin)), (min(img_w, xmax), min(img_h, ymax)), 1, -1)
                
            import torch
            car[f"{f_type}_mask_tensor"] = torch.from_numpy(mask_np).to(device=device, dtype=torch.bool)
            
        cars.append(car)

    # Assign Masks to Candidate Boxes
    for i in range(len(masks_tensor)):
        mask = masks_tensor[i]
        score = scores_tensor[i].item()
        is_rear = is_rear_flags[i]
        
        total_mask_pixels = pixel_counts[i]
        if total_mask_pixels < min_mask_area:
            continue
            
        # Centroid extraction
        y_coords, x_coords = torch.where(mask > 0)
        if len(x_coords) == 0: continue
        mask_cx = x_coords.float().mean().item()
        mask_cy = y_coords.float().mean().item()
            
        m3d = centroids_3d[i]
        candidates_2d = []
        candidates_3d = []
        box_padding = 0.2
        
        for car in cars:
            face_mask = car['rear_mask_tensor'] if is_rear else car['front_mask_tensor']
            pixels_in_box = torch.sum((mask > 0) & face_mask).item()
            
            if pixels_in_box > 0:
                candidates_2d.append({'car': car, 'containment': 0, 'pixels_in_box': pixels_in_box})
                    
            if m3d is not None:
                row = car['row']
                V_x, V_y = row['x_3d'], row['y_3d']
                yaw_val = row['yaw_3d'] if not pd.isna(row['yaw_3d']) else 0.0
                W = row.get('width_3d', 2.0)
                L = row.get('length_3d', 5.0)
                dx = m3d[0] - V_x
                dy = m3d[1] - V_y
                l_x = np.cos(yaw_val)*dx + np.sin(yaw_val)*dy
                l_y = -np.sin(yaw_val)*dx + np.cos(yaw_val)*dy
                
                # Check if the mask's 3D projection lands within the vehicle bounds
                if abs(l_x) <= (W/2 + box_padding) and abs(l_y) <= (L/2 + box_padding):
                    candidates_3d.append({'car': car, 'l_x': l_x, 'l_y': l_y})
                 
        # 3D - 2D Hybrid Resolution Logic
        ids_2d = [c['car']['track_id'] for c in candidates_2d]
        ids_3d = [c['car']['track_id'] for c in candidates_3d]
        unique_track_ids = set(ids_2d + ids_3d)
        num_candidates = len(unique_track_ids)
        
        winner_car = None
        state_tag = 2
        is_left_vehicle = False

        if num_candidates == 1:
            track_id = list(unique_track_ids)[0]
            if track_id in ids_3d:
                state_tag = 1
                c3d = next(c for c in candidates_3d if c['car']['track_id'] == track_id)
                winner_car = c3d['car']
                is_left_vehicle = c3d['l_y'] > 0
            else:
                state_tag = 0
                c2d = next(c for c in candidates_2d if c['car']['track_id'] == track_id)
                winner_car = c2d['car']
                
                # 2D assignment resolution based on bounding box geometry
                row = winner_car['row']
                c_str = row.get("rear_corners_2d" if is_rear else "front_corners_2d", None)
                valid_corners = False
                vehicle_left_cx, vehicle_right_cx = None, None
                
                if type(c_str) != float and c_str is not None:
                    try:
                        pts = np.array(ast.literal_eval(c_str)) if isinstance(c_str, str) else np.array(c_str)
                        if len(pts) == 4:
                            if is_rear:
                                vehicle_right_cx, vehicle_left_cx = (pts[0][0] + pts[2][0])/2, (pts[1][0] + pts[3][0])/2
                            else:
                                vehicle_left_cx, vehicle_right_cx = (pts[0][0] + pts[2][0])/2, (pts[1][0] + pts[3][0])/2
                            valid_corners = True
                    except: pass
                    
                if valid_corners:
                    is_left_vehicle = abs(mask_cx - vehicle_left_cx) < abs(mask_cx - vehicle_right_cx)
                else:
                    face_cx = row.get('rear_cx' if is_rear else 'front_cx', row['center_x'])
                    is_on_left_screen = mask_cx < face_cx
                    is_left_vehicle = (is_on_left_screen and is_rear) or (not is_on_left_screen and not is_rear)
                    
        elif num_candidates > 1:
            # Multi-candidate conflict resolution relying strictly on precise 3D placement
            if len(candidates_3d) == 1:
                winner = candidates_3d[0]
                winner_car = winner['car']
                state_tag = 1
                is_left_vehicle = winner['l_y'] > 0

        # Append winning candidate to track register
        if winner_car is not None:
            track_wins[winner_car['track_id']].append({
                'mask_idx': i,
                'score': score, 
                'mask_cx': mask_cx, 'mask_cy': mask_cy,
                'mask_cx_3d': m3d[0] if m3d is not None else None,
                'mask_cy_3d': m3d[1] if m3d is not None else None,
                'area': total_mask_pixels,
                'is_rear': is_rear, 'yaw_3d': winner_car['yaw_3d'],
                'cam_name': winner_car['cam_name'],
                'state_tag': state_tag,
                'is_left_vehicle': is_left_vehicle
            })

    # Geometric Left/Right Classification
    updated_rows = []
    for _, row in df_2d_boxes.iterrows():
        row_dict = row.to_dict()
        track_id = int(row['track_id'])
        claimed_masks = track_wins[track_id]
        
        row_dict['frame_generated_masks'] = num_generated
        row_dict['frame_valid_masks'] = num_valid
        
        row_dict['rear_left_indicator'] = None
        row_dict['rear_right_indicator'] = None
        row_dict['front_left_indicator'] = None
        row_dict['front_right_indicator'] = None
        row_dict['associated_masks'] = "[]"
        
        if claimed_masks:
            # Sort masks by SAM Score (Descending) to prioritize the best segmentation
            claimed_masks = sorted(claimed_masks, key=lambda x: x['score'], reverse=True)
            
            for m in claimed_masks:
                is_rear_target = m['is_rear'] 
                
                if m['state_tag'] == 1:
                    is_left_vehicle = m['is_left_vehicle']
                else:
                    c_str = row.get("rear_corners_2d" if is_rear_target else "front_corners_2d", None)
                    valid_corners = False
                    vehicle_left_cx, vehicle_right_cx = None, None

                    if type(c_str) != float and c_str is not None:
                        try:
                            pts = np.array(ast.literal_eval(c_str)) if isinstance(c_str, str) else np.array(c_str)
                            if len(pts) == 4:
                                # 3D Order mapping from transform.py:
                                if is_rear_target:
                                    vehicle_right_cx = (pts[0][0] + pts[2][0]) / 2.0
                                    vehicle_left_cx  = (pts[1][0] + pts[3][0]) / 2.0
                                else:
                                    vehicle_left_cx  = (pts[0][0] + pts[2][0]) / 2.0
                                    vehicle_right_cx = (pts[1][0] + pts[3][0]) / 2.0
                                valid_corners = True
                        except: pass

                    if valid_corners:
                        dist_to_vehicle_left = abs(m['mask_cx'] - vehicle_left_cx)
                        dist_to_vehicle_right = abs(m['mask_cx'] - vehicle_right_cx)
                        is_left_vehicle = dist_to_vehicle_left < dist_to_vehicle_right
                    else:
                        if is_rear_target and 'rear_cx' in row and not pd.isna(row['rear_cx']): face_cx = row['rear_cx']
                        elif not is_rear_target and 'front_cx' in row and not pd.isna(row['front_cx']): face_cx = row['front_cx']
                        else: face_cx = row['center_x']

                        is_on_left = m['mask_cx'] < face_cx
                        is_left_vehicle = (is_on_left and is_rear_target) or (not is_on_left and not is_rear_target)
                
                # Assign to 4 specific columns based on geometry
                if is_left_vehicle:
                    col_target = 'rear_left_indicator' if is_rear_target else 'front_left_indicator'
                else:
                    col_target = 'rear_right_indicator' if is_rear_target else 'front_right_indicator'

                if row_dict[col_target] is None:
                    row_dict[col_target] = str({
                        'mask_idx': m['mask_idx'],
                        'mask_cx': m['mask_cx'],
                        'mask_cy': m['mask_cy'],
                        'is_rear': is_rear_target,
                        'state_tag': m['state_tag'],
                        'cx_3d': m['mask_cx_3d'],
                        'cy_3d': m['mask_cy_3d']
                    })

            # Log data for all associated masks in a single column
            row_dict['associated_masks'] = str([
                {
                    'mask_idx': m['mask_idx'], 
                    'state_tag': m['state_tag'], 
                    'cx_3d': round(m['mask_cx_3d'], 2) if m['mask_cx_3d'] is not None else None,
                    'cy_3d': round(m['mask_cy_3d'], 2) if m['mask_cy_3d'] is not None else None,
                    'cx': m['mask_cx'], 
                    'cy': m['mask_cy'], 
                    'score': round(m['score'], 4),
                    'area': m['area']
                } for m in claimed_masks
            ])

        updated_rows.append(row_dict)

    return pd.DataFrame(updated_rows)

#### 4.4.3 Mask Vertical Elongation

**Function Overview: `expand_mask_vertical`**
To overcome special cases like SAM3 missing the indicators for cars with split lamp cluster with seperate indicator, this function employs a vertical padding to improve coverage of missed indicator 

*Inputs:*

* **`mask_array:`** A 2D binary numpy array representing the vehicle mask.
* **`expansion_ratio:`** The fraction of the mask's total height to append to the top and bottom.

*Core Mechanism:*

* **`Boundary Detection:`** It extracts the vertical min and max indices (the physical height) of the existing mask.
* **`Kernel Calculation:`** It dynamically calculates the pixel size based on the `expansion_ratio` and object height.
* **`Morphological Dilation:`** It applies a 1D vertical elongation (`cv2.dilate`) using a rectangular expansion to strictly elongate along the Y-axis without bleeding into lateral background noise.

*Outputs:*

* A new `uint8` binary mask array of the same shape, stretched vertically.

In [ ]:
def expand_mask_vertical(mask_array, expansion_ratio):
    """
    Expands a binary mask upward and downward only (width unchanged).
    
    Args:
        mask_array (np.ndarray): A 2D binary array representing the mask.
        expansion_ratio (float): Fraction of the mask's bounding-box height to add top and bottom.
        
    Returns:
        np.ndarray: A uint8 mask of the same shape with the vertical expansion applied.
    """

    if expansion_ratio <= 0:
        return (mask_array > 0).astype(np.uint8)
        
    mask_u8 = (mask_array > 0).astype(np.uint8)
    y_idx, _ = np.where(mask_u8 > 0)
    
    if len(y_idx) == 0:
        return mask_u8
        
    h = int(y_idx.max() - y_idx.min())
    k_size = max(1, int(h * expansion_ratio))
    
    # Kernel size must be odd for symmetric morphological dilation
    k_size = k_size if k_size % 2 == 1 else k_size + 1
    
    if k_size < 3:
        return mask_u8
        
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, k_size))
    return cv2.dilate(mask_u8, kernel, iterations=1)

### 4.5 Mask Amber Percent Computation

For each successfully associated and classified indicator mask, the pipeline must determine whether the light is physically ON or OFF in that specific camera frame. A naive approach using RGB brightness thresholds fails under real-world driving conditions due to sunshine overexposure in the day, or high luminance variability caused by approaching headlights during the night. To address this, we convert the masked pixel region into the **CIELAB** (L\*a\*b\*) color space.

CIELAB is a uniform color space, meaning that a fixed Euclidean distance between two points corresponds to an equal perceived color difference regardless of where in the color space those points lie, this is similar to how Humans percieve colour. This property makes it significantly more robust and easy to tune compared to HSV or RGB based approaches, where the definition of "amber" shifts dramatically under different exposure conditions. The three axes encode:

* **L\*:** Perceptual lightness (0 = black, 100 = white)
* **a\*:** Green–Red axis (negative = green, positive = red/magenta)
* **b\*:** Blue–Yellow axis (negative = blue, positive = yellow)

#### 4.5.1 Defining the Amber Target

Automotive amber indicator lights occupy a consistent, tightly bounded cluster in CIELAB space. We define this cluster by a calibrated center point $(L^*_{ideal},\, a^*_{ideal},\, b^*_{ideal})$, determined empirically from labeled indicator samples across the dataset and taking referencing from ECE standards[[x]][#ref]. Any pixel within the mask whose CIELAB coordinates fall within a spherical radius $r_{max}$ of this ideal point is classified as an amber pixel:

$$
d_i = \sqrt{(L^*_i - L^*_{ideal})^2 + (a^*_i - a^*_{ideal})^2 + (b^*_i - b^*_{ideal})^2}
$$

$$
d_i < r_{max} \implies \text{Amber candidate}
$$

#### 4.5.2 Yellowness Floor Condition

The spherical boundary alone is insufficient to distinguish amber from certain shades of bright white (brake light bleed-over) and saturated red, which can produce similar Euclidean distances when luminance varies. To eliminate this ambiguity, we apply a hard cutoff constraint on the `b*` axis:

$$
b^*_i \geq b^*_{min} \implies \text{Confirmed Amber}
$$

Since positive `b*` quantifies the degree of yellowness directly, requiring $b^* \geq b^*_{min}$ guarantees that the candidate pixel is inherently yellow—not just close in distance due to elevated luminance.

#### 4.5.3 Final Classification

A pixel is declared amber if and only if both conditions are satisfied:

$$
\text{Amber} \iff d_i < r_{max} \;\; \text{AND} \;\; b^*_i \geq b^*_{min}
$$

<div align="center">
  <figure>
    <img src="images/cielab-space.png" width="700" alt="CIELAB Boundary">
    <figcaption><b>Fig. 8:</b> CIELAB boundary based color chart</figcaption>
  </figure>
</div>

The proportion of amber-classified pixels within the valid mask area gives the primary output signal, `amber_pct`, which forms the time-series input to the next blink detection algorithm:

$$
\text{amber_pct} = \frac{\text{amber pixel count}}{\text{total valid mask pixels}} \times 100
$$

Note that "valid" pixels exclude fully black pixels ($R=G=B=0$), which arise from the mask boundary and compression artifacts rather than genuine surface reflections.

**Function Overview: `extract_cielab_amber_metrics`**
This function computes the intensity of amber/yellow light inside a specific mask by projecting pixels into the CIELAB color space and applying spherical thresholding along with a cutoff.

*Inputs:*

* **`image_rgb:`** The 2D RGB image array.
* **`mask_array:`** The binary mask identifying the region of interest.
* **`ideal_l, ideal_a, ideal_b:`** The target center coordinates of amber light in CIELAB space.
* **`max_dist:`** The maximum Euclidean distance permitted from the ideal CIELAB center.
* **`b_min:`** The minimum threshold for the `b*` channel (yellowness) to reject red/white noise.

*Core Mechanism:*

* **`Color Space Conversion:`** It isolates valid, non-black RGB pixels within the mask and converts them into the CIELAB color space.
* **`Euclidean Distance Calculation:`** It computes the 3D Euclidean distance of every valid pixel from the user-defined `ideal_L, ideal_a, ideal_b` focal point.
* **`Dual Thresholding:`** It classifies a pixel as active amber if its distance is within `max_dist` AND its `b*` value exceeds `b_min` (ensuring the pixel is fundamentally yellow).

*Outputs:*

* **`amber_pct:`** The percentage of amber pixels relative to the total valid mask area (used as the primary time-series signal).
* **`amber_count, total_px:`** Raw counts of amber and total pixels for diagnostic logging.

<div align="center">
  <figure>
    <img src="images/amber-percent.png" width="1000" alt="Amber PCT">
    <figcaption><b>Fig. 9:</b> Amber percent calculation example for brake lights lit rear taillight</figcaption>
  </figure>
</div>

In [ ]:
def extract_cielab_amber_metrics(image_rgb, mask_array,
                                  ideal_l, ideal_a, ideal_b,
                                  max_dist, b_min):
    """
    Counts amber pixels inside a mask using a CIELAB sphere and b* floor threshold.
    
    Args:
        image_rgb (np.ndarray): The source RGB image array.
        mask_array (np.ndarray): The 2D binary mask denoting the region to check.
        ideal_l (float): Target L* value for the amber color.
        ideal_a (float): Target a* value for the amber color.
        ideal_b (float): Target b* value for the amber color.
        max_dist (float): Maximum 3D Euclidean distance in LAB space to be considered amber.
        b_min (float): Minimum b* (yellowness) floor to eliminate false positives.

    Returns:
        tuple: (amber_pct, amber_count, total_px)
            - amber_pct (float): Amber pixels as % of total mask pixels [primary blink signal].
            - amber_count (int): Raw amber pixel count [diagnostic].
            - total_px (int): Total non-black mask pixels [diagnostic / denominator].
    """

    valid_pixels = image_rgb[mask_array > 0]
    non_black    = np.any(valid_pixels > 0, axis=1)
    valid_pixels = valid_pixels[non_black]
    total_px     = len(valid_pixels)

    if total_px == 0:
        return (0.0, 0, 0)

    lab    = cv2.cvtColor(valid_pixels.reshape(-1, 1, 3),
                          cv2.COLOR_RGB2LAB).reshape(-1, 3)
    L_vals = lab[:, 0].astype(float)
    a_vals = lab[:, 1].astype(float)
    b_vals = lab[:, 2].astype(float)

    # Plain 3D Euclidean distance in LAB space
    dist = np.sqrt(
        (L_vals - ideal_l) ** 2 +
        (a_vals - ideal_a) ** 2 +
        (b_vals - ideal_b) ** 2
    )

    amber_mask  = (dist < max_dist) & (b_vals >= b_min)
    amber_count = int(amber_mask.sum())
    amber_pct   = amber_count / total_px * 100.0

    return (amber_pct, amber_count, total_px)

### 4.6 Main Pipeline


**Function Overview: `process_all_valid_images_with_sam`**

This function loops through all camera images and obtains indicator data for track ids.

<div align="center">
  <figure>
    <img src="images/main-pipeline.png" width="800" alt="Association Logic">
    <figcaption><b>Fig. 10:</b> Logic flow diagram for main pipeline</figcaption>
  </figure>
</div>


*Inputs:*

*Data & Calibration*
* **`camera_dataframes:`** df_2d_boxes
* **`sync_table:`** lookup table from dataset for temporal alignment between sensors.
* **`calib_data:`** Sensor calibration dictionaries containing intrinsic camera matrices (K), extrinsic LiDAR-to-camera matrices (vTl, cTv), and lens undistortion functions.
* **`data_root:`** The master directory path where .jpg images and .npz LiDAR files are stored.

*Models & Hardware*
* **`processor, model, device:`** The instantiated SAM 3 processing pipeline and hardware (e.g., CUDA).
* **`depth_processor, depth_model:`** The instantiated DA3 processing pipeline.

*Inference Configurations*
* **`inference_targets:`** Positive text prompt for SAM 3.
* **`exclude_prompt:`** An optional negative text prompt for SAM 3.
* **`target_object_types:`** A list of vehicle classes to filter the input data (e.g., ['Car', 'Truck', 'Bus']).

*Thresholds & Hyperparameters*
* **`sam_conf_threshold, sam_mask_threshold:`** Confidence limits for generating the segmentation masks.
* **`min_mask_area:`** The absolute minimum pixel count required to keep a mask (filters out microscopic noise).
* **`ideal_l, ideal_a, ideal_b:`** Calibrated CIELAB target coordinates for amber.
* **`rear_max_dist, rear_b_min, rear_expansion:`** CIELAB thresholds and vertical expansion ratios specifically tuned for rear taillights.
* **`front_max_dist, front_b_min, front_expansion:`** CIELAB thresholds and vertical expansion ratios specifically tuned for front headlights.

*Input-Output Flags*
* **`save_csv:`** Boolean flag. If True, periodically saves backup checkpoints of the DataFrame during long runs and outputs a final .csv in "runs" folder.
* **`save_masks:`** Boolean flag. If True, exports the raw 2D boolean masks to compressed .npz archive files in "runs" folder.

*Core Mechanism:*

* **`Dense Depth Generation (Step 1):`** Projects raw LiDAR point clouds into 2D camera space and uses Depth Anything 3 (DA3) to generate a continuous, dense absolute depth map (`dense_cam_z`).
* **`Mask Generation (Step 2):`** Prompts the Segment Anything 3 (SAM 3) model to generate pixel-perfect boolean masks for candidate indicators.
* **`3D Centroid Calculation (Step 3):`** Fuses the boolean masks with the dense depth map to calculate the median depth for each indicator, which is then projected back into 3D space (`mask_centroids_3d`).
* **`Mask Association (Step 4):`** Cross-references the 3D mask centroids and 2D bounding boxes against the known vehicle tracks to successfully assign indicators to specific vehicles (`associate_masks_to_boxes`).
* **`Photometric Extraction (Step 5):`** Vertically expands the winning masks (`expand_mask_vertical`) and evaluates the pixels against the CIELAB color sphere (`extract_cielab_amber_metrics`) to determine the final amber counts and percentages.

*Outputs:*

*Return Value*
* **`df_final_master (Pandas DataFrame):`** The result of the pipeline. It takes the original camera_dataframes and successfully appends all the geometric and photometric features of indicators into the target vehicles.

*Optional Outputs*
* **`Checkpoints (SAM3_hybrid_checkpoint.csv):`** Intermediate data saved every 200 images to prevent data loss.
* **`Final Export (SAM3_hybrid_final.csv):`** export of df_final_master DataFrame in csv format.
* **`Mask Archives (.npz files):`** Saved in runs/masks/<camera_name>/<timestamp>.npz, storing all generated segmentation masks in zipped format.

In [ ]:
def process_all_valid_images_with_sam(
    camera_dataframes, 
    sync_table,     
    calib_data,
    data_root, 
    processor, 
    model, 
    device, 
    inference_targets=None,
    exclude_prompt=None,          
    sam_conf_threshold=0.50,      
    sam_mask_threshold=0.80,         
    min_mask_area=20,
    target_object_types=['Car', 'Van', 'Truck', 'Bus', 'Trailer', 'E-Scooter', 'Motorcycle'],
    save_csv=False,
    save_masks=False,
    depth_processor=None,
    depth_model=None,
    ideal_l=205,
    ideal_a=153,
    ideal_b=208,
    rear_max_dist=80,
    rear_b_min=135,
    rear_expansion=0.20,
    front_max_dist=80,
    front_b_min=135,
    front_expansion=0.10,
):
    """
    Master pipeline orchestrator for semantic mask generation, depth fusion, and association.
    
    Iterates through all valid vehicle timestamps across cameras, generates masks using SAM3,
    fuses LiDAR point clouds for dense depth estimation, and associates detected indicator 
    masks back to their respective vehicle tracks. Finally, it extracts CIELAB color metrics
    for the matched regions.
    
    Args:
        camera_dataframes (dict or pd.DataFrame): Track-wise or frame-wise vehicle detections.
        sync_table (dict): Lookup table to match master timestamps to camera image timestamps.
        calib_data (dict): Calibration parameters for camera undistortion and 3D projection.
        data_root (str): Root directory containing the raw image and LiDAR datasets.
        processor: The main SAM3 processor instance.
        model: The main SAM3 model instance.
        device: Hardware device to run the inference on.
        inference_targets (list of dict, optional): Targets to search for (e.g., headlights).
        exclude_prompt (str, optional): Text prompt to explicitly ignore during segmentation.
        sam_conf_threshold (float): Confidence threshold for SAM3 mask generation.
        sam_mask_threshold (float): Threshold for binarizing SAM3 outputs.
        min_mask_area (int): Minimum pixel area for valid masks.
        target_object_types (list of str): List of vehicle class names to process.
        save_csv (bool): If True, intermediate and final results are saved to disk.
        save_masks (bool): If True, generated masks are compressed and saved to disk.
        depth_processor: Processor for the dense depth estimation model.
        depth_model: Model for dense depth estimation.
        ideal_l (float): Target L* value for the indicator color.
        ideal_a (float): Target a* value for the indicator color.
        ideal_b (float): Target b* value for the indicator color.
        rear_max_dist (float): Max LAB space distance for rear indicators.
        rear_b_min (float): Min b* value for rear indicators.
        rear_expansion (float): Vertical expansion ratio for rear indicator masks.
        front_max_dist (float): Max LAB space distance for front indicators.
        front_b_min (float): Min b* value for front indicators.
        front_expansion (float): Vertical expansion ratio for front indicator masks.
        
    Returns:
        pd.DataFrame: A unified dataframe containing all tracked vehicles with 
                      matched indicator masks and their respective color metrics.
    """

    # Initialising Variables
    output_dir = "runs/result"
    dataset_name = os.path.basename(os.path.normpath(data_root)) or "dataset"
    checkpoint_interval = 200
    
    # Safely default to targets if none are provided
    if inference_targets is None:
        inference_targets = [
            {"text": "car taillight", "is_rear": True},
            {"text": "car headlight", "is_rear": False}
        ]

    print("="*60)
    print(f"Starting the pipeline....")
    print(f"Target Classes: {target_object_types}") 
    print(f"Segmentation Targets: {[t['text'] for t in inference_targets]}") 
    print("="*60)
    
    # Filter given dataset for target objects
    if isinstance(camera_dataframes, dict):
        valid_dfs = []
        for cam, df in camera_dataframes.items():
            if not df.empty:
                df_filtered = df[df['object_type'].isin(target_object_types)].copy()
                if not df_filtered.empty:
                    if 'camera' not in df_filtered.columns:
                        df_filtered['camera'] = cam
                    valid_dfs.append(df_filtered)
        if not valid_dfs: return pd.DataFrame()
        df_2d_boxes = pd.concat(valid_dfs, ignore_index=True)
    else:
        df_2d_boxes = camera_dataframes.copy()
        if not df_2d_boxes.empty and 'object_type' in df_2d_boxes.columns:
            df_2d_boxes = df_2d_boxes[df_2d_boxes['object_type'].isin(target_object_types)]

    if df_2d_boxes.empty: return pd.DataFrame()

    # Drop a row if valid image is not present in the folder
    def get_image_timestamp(row):
        try:
            ts_idx = sync_table['timestamp_nanoseconds'].index(row['timestamp'])
            return int(sync_table[row['camera']][ts_idx][0])
        except (ValueError, IndexError, KeyError):
            return None

    df_2d_boxes['image_associated_timestamp'] = df_2d_boxes.apply(get_image_timestamp, axis=1)
    df_2d_boxes = df_2d_boxes.dropna(subset=['image_associated_timestamp'])
    
    if df_2d_boxes.empty: return pd.DataFrame()
    df_2d_boxes['image_associated_timestamp'] = df_2d_boxes['image_associated_timestamp'].astype(int)

    # Image List Generation
    all_associations = []
    unique_images = df_2d_boxes[['camera', 'timestamp', 'image_associated_timestamp']].drop_duplicates()
    
    # Looping via unique images
    for index, row in tqdm(unique_images.iterrows(), total=len(unique_images), desc="Processing Images"):
        cam = row['camera']
        master_ts = row['timestamp']
        img_time = row['image_associated_timestamp'] 
        
        df_cars_in_image = df_2d_boxes[(df_2d_boxes['camera'] == cam) & (df_2d_boxes['timestamp'] == master_ts)]
        if df_cars_in_image.empty: 
            continue

        image_path = os.path.join(data_root, cam, f"{img_time}.jpg") 
        
        # Preserve track ids with blank stats if image is missing
        if not os.path.exists(image_path): 
            df_blank = associate_masks_to_boxes(df_cars_in_image, [], [], [], [], [])
            all_associations.append(df_blank)
            continue 
        
        raw_img = cv2.imread(image_path)
        
        # Preserve track ids with blank stats if image is corrupted
        if raw_img is None: 
            df_blank = associate_masks_to_boxes(df_cars_in_image, [], [], [], [], [])
            all_associations.append(df_blank)
            continue
            
        img_rgb = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
        
        # Fisheye lens distorted image is flattened using native function from 'DrivIng'
        undistort_function = calib_data['cameras'][cam]['undistort_function']
        flat_img = undistort_function(img_rgb)
        pil_image = Image.fromarray(flat_img)

        
        # Step 1: generating 3d dense depth map using LiDAR pcd and Depth Anything 3
        lidar_path = os.path.join(data_root, "middle_lidar", f"{master_ts}.npz")
        valid_lidar = os.path.exists(lidar_path)

        K = None
        lTc = None
        dense_cam_z = None
        
        # Transforming lidar 3d data to 2d and running DA3
        if valid_lidar and depth_model is not None:
            
            img_h, img_w = flat_img.shape[:2]
            
            u_valid, v_valid, true_z, K, lTc = project_lidar_to_2d(
                lidar_path=lidar_path, 
                calib_data=calib_data, 
                camera_name=cam, 
                img_w=img_w, 
                img_h=img_h)
            
            dense_cam_z = get_dense_metric_depth(flat_img, u_valid, v_valid, true_z, depth_processor, depth_model, device)


        # Step 2: generating SAM3 masks for an image
        all_masks_tensors = []
        all_scores_tensors = []
        all_is_rear_flags = []
        
        for target in inference_targets:
            masks_t, scores_t = generate_sam3_masks(
                image_path=pil_image, 
                text_prompt=target["text"], 
                exclude_prompt=exclude_prompt,   
                processor=processor, model=model, device=device, 
                conf_threshold=sam_conf_threshold, mask_threshold=sam_mask_threshold
            )
            
            if len(masks_t) > 0:
                all_masks_tensors.append(masks_t)
                all_scores_tensors.append(scores_t)
                all_is_rear_flags.extend([target["is_rear"]] * len(masks_t))
                
        # Preserve cars with blank stats if SAM finds zero masks
        if len(all_masks_tensors) == 0:
            df_blank = associate_masks_to_boxes(df_cars_in_image, [], [], [], [], [])
            all_associations.append(df_blank)
            continue
            
        # Concat GPU Tensors
        master_masks_tensor = torch.cat(all_masks_tensors, dim=0)
        master_scores_tensor = torch.cat(all_scores_tensors, dim=0)

        
        # Step 3: logging pixel counts + 3D centroids
        mask_pixel_counts = []
        num_valid_masks = 0
        
        masks_cpu = master_masks_tensor.cpu().numpy()
        
        if save_masks == True:
            # Create exact requested structure: runs/masks_<dataset_name>/<camera_name>/
            mask_save_dir = os.path.join(output_dir, f"masks_{dataset_name}", cam)
            os.makedirs(mask_save_dir, exist_ok=True)
            
            # Dictionary comprehension to name each mask by its index
            mask_dict = {str(i): (m_np > 0) for i, m_np in enumerate(masks_cpu)}
            
            if mask_dict:
                # Save ALL masks into ONE highly compressed .npz archive named after the timestamp
                archive_path = os.path.join(mask_save_dir, f"{img_time}.npz")
                np.savez_compressed(archive_path, **mask_dict)
        
        mask_centroids_3d = []
        for m_np in masks_cpu:
            # Pixel count is free: just sum the boolean mask
            total_px = int(m_np.sum())
            mask_pixel_counts.append(total_px)
            
            if total_px < min_mask_area:
                mask_centroids_3d.append(None)
                continue
            
            num_valid_masks += 1
            
            median_3d = None
            if dense_cam_z is not None and K is not None and lTc is not None:
                y_indices, x_indices = np.where(m_np > 0)
                if len(x_indices) > 0:
                    cx_pixel = int(np.mean(x_indices))
                    cy_pixel = int(np.mean(y_indices))
                    mask_depths = dense_cam_z[m_np > 0]
                    if len(mask_depths) > 0:
                        median_cam_z = np.median(mask_depths)
                        X_c = (cx_pixel - K[0,2]) * median_cam_z / K[0,0]
                        Y_c = (cy_pixel - K[1,2]) * median_cam_z / K[1,1]
                        cam_point_3d = np.array([X_c, Y_c, median_cam_z, 1.0])
                        median_3d = (lTc @ cam_point_3d)[:3]

            mask_centroids_3d.append(median_3d)


        # Step 4: associating masks to track ids & classification to left-right
        df_verified = associate_masks_to_boxes(
            df_2d_boxes=df_cars_in_image, 
            masks_tensor=master_masks_tensor, 
            scores_tensor=master_scores_tensor,
            is_rear_flags=all_is_rear_flags,  
            pixel_counts=mask_pixel_counts, 
            centroids_3d=mask_centroids_3d,
            min_mask_area=min_mask_area,
            num_generated=len(master_masks_tensor),    
            num_valid=num_valid_masks    
        )


        # Step 5: vertical expansion + CIELAB amber % for assoicated-classified masks
        INDICATOR_COLS = ['rear_left_indicator', 'rear_right_indicator',
                           'front_left_indicator', 'front_right_indicator']

        # Collect {mask_idx: is_rear} for every assigned indicator slot
        winning = {}   # mask_idx -> is_rear (bool)
        for col in INDICATOR_COLS:
            if col not in df_verified.columns:
                continue
            col_is_rear = col.startswith('rear_')
            for val in df_verified[col].dropna():
                try:
                    winning[ast.literal_eval(str(val))['mask_idx']] = col_is_rear
                except: pass

        # Run vertical expansion + CIELAB filtering
        metrics_cache = {}
        for midx, is_rear in winning.items():
            if not (0 <= midx < len(masks_cpu)):
                continue
            exp   = rear_expansion if is_rear else front_expansion
            mdist = rear_max_dist  if is_rear else front_max_dist
            bmin  = rear_b_min     if is_rear else front_b_min

            exp_mask = expand_mask_vertical(masks_cpu[midx], exp)
            metrics_cache[midx] = extract_cielab_amber_metrics(
                flat_img, exp_mask,
                ideal_l, ideal_a, ideal_b,
                mdist, bmin
            )

        # Patch amber_pct / amber_count / total_px back into each column
        for col in INDICATOR_COLS:
            if col not in df_verified.columns:
                continue
            for idx, val in df_verified[col].items():
                if val is None or (isinstance(val, float) and pd.isna(val)):
                    continue
                try:
                    d = ast.literal_eval(str(val))
                    midx = d['mask_idx']
                    if midx in metrics_cache:
                        d['amber_pct'], d['amber_count'], d['total_px'] = metrics_cache[midx]
                        df_verified.at[idx, col] = str(d)
                except: pass

        if not df_verified.empty:
            all_associations.append(df_verified)

        
        # Intermediate csv outputs to prevent data loss in case of server failure
        if save_csv and len(all_associations) > 0 and len(all_associations) % checkpoint_interval == 0:
            os.makedirs(output_dir, exist_ok=True)
            chkpt_df = pd.concat([d.dropna(axis=1, how='all') for d in all_associations], ignore_index=True)
            chkpt_path = os.path.join(output_dir, f"SAM3_hybrid_checkpoint_{dataset_name}.csv")
            chkpt_df.to_csv(chkpt_path, index=False)
            tqdm.write(f"CHECKPOINT: Flushed {len(chkpt_df)} rows to {chkpt_path}")

    # Master DataFrame Stitching
    if all_associations:
        cleaned_associations = [df.dropna(axis=1, how='all') for df in all_associations]
        
        df_final_master = pd.concat(cleaned_associations, ignore_index=True)
        
        df_final_master = df_final_master.sort_values(by=['timestamp', 'camera', 'track_id']).reset_index(drop=True)
        
        if 'image_associated_timestamp' in df_final_master.columns:
            df_final_master = df_final_master.drop(columns=['image_associated_timestamp'])
        
        if save_csv:
            os.makedirs(output_dir, exist_ok=True)
            final_path = os.path.join(output_dir, f"SAM3_hybrid_final_{dataset_name}.csv")
            df_final_master.to_csv(final_path, index=False)
            print(f"FINAL SAVED: {final_path}")
            
        print("\n PIPELINE COMPLETE!")
        return df_final_master
    else:
        print("\n PIPELINE COMPLETE: No matches found.")
        return pd.DataFrame()

**Pipeline Execution:** 
Processing the sample dataset (Estimated time: 7 - 10 minutes).

**Fast-Track:** 
To skip this execution step and load the existing results, bypass this cell and execute the next cell to load the DataFrame via CSV.

In [ ]:
# Start the high-precision stopwatch
start_time = time.perf_counter()

# Define your Prompts here at the top level
MY_TARGETS = [
    {"text": "car taillight", "is_rear": True},
    {"text": "car headlight", "is_rear": False}
]

# Configure Tunable Parameters
# day parameters     rear_max_dist=80, rear_b_min=135, rear_expansion=0.20, front_max_dist=80, front_b_min=155, front_expansion=0.20
# night parameters     rear_max_dist=80, rear_b_min=170, rear_expansion=0.20, front_max_dist=80, front_b_min=180, front_expansion=1.00

target_object_types = ['Car', 'Van', 'Truck', 'Bus']

# Execute the Hybrid Tensor Pipeline
df_final_results = process_all_valid_images_with_sam(
    camera_dataframes=camera_dataframes,   
    sync_table=sync_table,  
    calib_data=calib_data,
    data_root=seq_dir, 
    processor=processor, 
    model=model, 
    device=device,
    inference_targets=MY_TARGETS,       
    sam_conf_threshold=0.5,      
    sam_mask_threshold=0.60,        
    min_mask_area=25,
    target_object_types=target_object_types,
    save_csv=True,
    save_masks=False,
    depth_processor=None,
    depth_model=model_DA3,
    ideal_l=205,
    ideal_a=153,
    ideal_b=208,
    rear_max_dist=80, 
    rear_b_min=170, 
    rear_expansion=0.20, 
    front_max_dist=80, 
    front_b_min=180, 
    front_expansion=1.00
)    

# Stop the stopwatch and calculate
end_time = time.perf_counter()
total_time = end_time - start_time
print(f"\n PIPELINE TIME: {total_time:.2f} seconds ({total_time / 60:.2f} minutes)")

*To load output directly from csv, uncomment and run this cell.*

In [ ]:
# df_final_results = pd.read_csv('topic-17-files/data_night_sample_result.csv')

In [ ]:
pd.set_option('display.max_columns', None) 
display(df_final_results.head(10))

### 4.7 Blinking State Estimation

Once the amber intensity is extracted as a continuous time-series signal (`amber_pct`), the pipeline must mathematically prove the presence of a periodic blinking pattern. Hence we use a blinking frequency based stratergy to overcome the shortfalls of colour based stratergy which cannot analyse dynamically overexposed indicators. 

**4.7.11 Rolling median and Detrending**

Before frequency analysis, we must isolate the high-frequency blink pulses from low-frequency environmental noise, such as a vehicle entering a shadow or an approaching headlight increasing the overall scene brightness or brakelight activation bleed. We achieve this by applying a rolling median filter to compute a dynamic baseline, which is then subtracted from the raw signal. This detrends the signal precisely at zero, isolating the rhythmic blink amplitudes.

Note: with the capture rate of 10 fps across the dataset sensors and according to ECE mandate that states that each blink must last for a minimum of 300ms, we can expect at least 2-3 frames to have captured a blink (each frame is ~100ms).
    
<div align="center">
  <figure>
    <img src="images/blinking-detrending.png" width="600" alt="blinking">
    <figcaption><b>Fig. 11:</b> Rolling median and detrending representative plot</figcaption>
  </figure>
</div>

**4.7.12 Autocorrelation and Frequency Validation**

To detect periodicity - due to varying manufacture sepcific blinking frequency - without assuming a set waveform, we compute the autocorrelation of the detrended signal. Autocorrelation measures the similarity of the signal against a time-shifted version of itself. If a vehicle is blinking, the autocorrelation function will exhibit a distinct peak at a time lag corresponding exactly to the blink period. 

We search for this peak within a restricted lag window corresponding to the ECE specific legal regulatory limits (1 Hz to 2 Hz) for automotive turn indicators, in this case we have set the windown between 0.8 Hz to 2.5 Hz rather than the exact ECE limits, to ensure slight padding effect for better signal capture. If a strong peak is found within this frequency band, the signal passes the initial periodicity check.

<div align="center">
  <figure>
    <img src="images/blinking-autocorrelation.png" width="600" alt="blinking">
    <figcaption><b>Fig. 12:</b> Autocorrelation function representative plot</figcaption>
  </figure>
</div>

**4.7.13 Binarization and De-noising**

To binarize and make the above signal into a square waveform, we apply a 3x pass filter.    

(i) Any peak below a set threshold of 0.3 * standard deviation of detrended signal is flattened.
    
(ii) Any 'OFF' gaps shorter than one frame (100ms) is merged with adjacent peaks. (Removes small dips between two peaks)
    
(iii) Any 'ON' spikes shorter than one frame (100ms) are tagged as 'OFF' (removes single-frame noise reflections)
    
<div align="center">
  <figure>
    <img src="images/blinking-denoising.png" width="600" alt="blinking">
    <figcaption><b>Fig. 13:</b> Binarization and de-noising representative plot</figcaption>
  </figure>
</div>

**4.7.14 Temporal Rejection**

Finally, the pipeline groups consecutive pulses into unified blinking "episodes". If the gap between two pulses exceeds 1.5 times the detected blink period (`1.5 * best_lag`), the episode is split. This ensures that disconnected artifacts or separate lane-change maneuvers are not incorrectly merged, allowing for the precise rejection of short bursts that fail to meet the minimum required cycle counts.
    
<center>
    <img src="images/blinking-segmentation.png" width="600" alt="blinking">
<center>
<div align="center">
  <figure>
    <img src="images/blinking-segmentation.png" width="600" alt="blinking">
    <figcaption><b>Fig. 14:</b> Blinking episodes representative plot</figcaption>
  </figure>
</div>

**Function Overview: `detect_blink_signal`**
This function implements detects periodic blinking patterns from a noisy array of amber pixel percentages.

*Inputs:*

* **`signal_pct:`** 1D array of amber percentages over time.
* **`fps:`** The camera frame rate.
* **`detrend_window_sec, detrend_noise_factor, min_detrended_range:`** Parameters controlling baseline subtraction and signal flatness gating.
* **`autocorr_min_peak, blink_min_hz, blink_max_hz:`** Parameters for periodicity and frequency validation.
* **`min_on_frames, min_off_frames, min_cycles, max_jitter_pct:`** Hyperparameters for temporal cleaning and rhythm stability.

*Core Mechanism:*

* **`Rolling-Median Detrending:`**
* **`Autocorrelation:`**
* **`Binarization & Denoising`** 
* **`Temporal Validation`**

*Outputs:*

* A dictionary containing the `is_blinking` boolean verdict, exact ON/OFF `frame_state` arrays, frequency calculations, and diagnostic strings.

In [ ]:
def detect_blink_signal(signal_pct, fps,
                         detrend_window_sec, detrend_noise_factor,
                         min_detrended_range,
                         autocorr_min_peak,
                         min_on_frames, min_off_frames,
                         blink_min_hz, blink_max_hz,
                         min_cycles, max_jitter_pct):
    """
    Detrend + Autocorrelation blink detection on a 1D amber-% signal.

    Stage 1: Rolling-median baseline subtraction (removes slow drift from
             illumination changes, vehicle movement, etc.).
    Stage 2: Flatness check on the detrended signal.
    Stage 3: Autocorrelation — shape-agnostic periodic detector.
    Stage 4: Binarize detrended signal for precise edge timing.
    Stage 5: Inter-pulse spacing (secondary confirmation).
    Stage 6: Final decision: freq_ok AND autocorr_strength_ok AND cycles_ok.
    """
    
    def run_length_clean(arr, min_on, min_off):
        """
        Two-pass run-length denoiser.
        Pass 1: fill short OFF gaps  (fixes notches at the top of pulses).
        Pass 2: remove short ON pulses (removes single-frame noise spikes).
        """
        n, out = len(arr), arr.copy()
        i = 0
        while i < n:                       # Pass 1: fill OFF gaps
            if out[i] == 0:
                j = i + 1
                while j < n and out[j] == 0:
                    j += 1
                if (j - i) < min_off and i > 0 and j < n:
                    out[i:j] = 1
                i = j
            else:
                i += 1
        i = 0
        while i < n:                       # Pass 2: remove short ON pulses
            if out[i] == 1:
                j = i + 1
                while j < n and out[j] == 1:
                    j += 1
                if (j - i) < min_on:
                    out[i:j] = 0
                i = j
            else:
                i += 1
        return out

    n_sig = len(signal_pct)

    # ── Stage 1: Rolling-median detrend ───────────────────────────────────────
    window = max(5, int(fps * detrend_window_sec))
    if window % 2 == 0:
        window += 1
    baseline  = pd.Series(signal_pct).rolling(
        window=window, center=True, min_periods=1
    ).median().values
    detrended = signal_pct - baseline

    det_p5    = np.percentile(detrended, 5)
    det_p95   = np.percentile(detrended, 95)
    det_range = float(det_p95 - det_p5)
    det_std   = float(np.std(detrended))

    # Base result dict
    result = dict(
        baseline=baseline, detrended=detrended,
        det_range=det_range, det_std=det_std,
        binary=np.zeros(n_sig, dtype=int),
        cleaned=np.zeros(n_sig, dtype=int),
        rising_edges=np.array([]), falling_edges=np.array([]),
        on_durations=np.array([]), spacings=np.array([]),
        autocorr=np.array([]), autocorr_lags=(0, 0),
        best_lag=None, autocorr_strength=0.0,
        median_spacing=None, mad_sp=None, jitter_pct=None,
        detected_hz=None, is_blinking=False, reason='', n_cycles=0,
        det_threshold=0.0,
        frame_state=np.full(n_sig, 'OFF', dtype=object),
    )

    # ── Stage 2: Flatness gate ─────────────────────────────────────────────────
    if det_range < min_detrended_range:
        result['reason'] = (
            f"Flat after detrending — detrended range {det_range:.1f}% "
            f"(need >= {min_detrended_range}%)"
        )
        return result

    # ── Stage 3: Autocorrelation ───────────────────────────────────────────────
    sig_c = detrended - detrended.mean()
    norm  = float(np.sum(sig_c ** 2))
    if norm < 1e-10:
        result['reason'] = "Signal energy is zero after detrending"
        return result

    autocorr_full = np.correlate(sig_c, sig_c, mode='full')
    autocorr      = autocorr_full[len(autocorr_full) // 2:]
    autocorr      = autocorr / autocorr[0]     # Normalise to [0, 1]
    result['autocorr'] = autocorr

    min_lag = max(2, int(np.floor(fps / blink_max_hz)))
    max_lag = min(len(autocorr) - 1, int(np.ceil(fps / blink_min_hz)) + 2)
    result['autocorr_lags'] = (min_lag, max_lag)

    search_region = autocorr[min_lag:max_lag + 1]
    ac_peaks, ac_props = find_peaks(search_region, height=autocorr_min_peak)

    if len(ac_peaks) == 0:
        result['reason'] = (
            f"No periodic peak in autocorrelation "
            f"(searched lags {min_lag}–{max_lag}, "
            f"need height >= {autocorr_min_peak})"
        )
        return result

    peaks_adj   = ac_peaks + min_lag
    best_idx    = np.argmax(ac_props['peak_heights'])
    best_lag    = int(peaks_adj[best_idx])
    ac_strength = float(ac_props['peak_heights'][best_idx])
    detected_hz = float(fps / best_lag)

    result['best_lag']          = best_lag
    result['autocorr_strength'] = ac_strength
    result['detected_hz']       = detected_hz

    # ── Stage 4: Binarize detrended for edge timing ────────────────────────────
    det_threshold = det_std * detrend_noise_factor
    result['det_threshold'] = float(det_threshold)
    binary  = (detrended > det_threshold).astype(int)
    cleaned = run_length_clean(binary, min_on_frames, min_off_frames)
    result['binary']  = binary
    result['cleaned'] = cleaned

    diff          = np.diff(cleaned, prepend=0)
    rising_edges  = np.where(diff ==  1)[0]
    falling_edges = np.where(diff == -1)[0]
    result['rising_edges']  = rising_edges
    result['falling_edges'] = falling_edges

    on_durations = []
    for r in rising_edges:
        fe = falling_edges[falling_edges > r]
        on_durations.append(int(fe[0] - r) if len(fe) else int(n_sig - r))
    result['on_durations'] = np.array(on_durations)

    # ── Stage 5: Edge spacing (secondary confirmation) ─────────────────────────
    n_pulses = len(rising_edges)
    n_cycles = n_pulses - 1 if n_pulses >= 2 else 0
    result['n_cycles'] = n_cycles

    if n_pulses >= 2:
        spacings   = np.diff(rising_edges).astype(float)
        median_sp  = float(np.median(spacings))
        mad_sp     = float(np.median(np.abs(spacings - median_sp)))
        jitter_pct = (mad_sp / median_sp * 100) if median_sp > 0 else 999.0
        result['spacings']       = spacings
        result['median_spacing'] = median_sp
        result['mad_sp']         = mad_sp
        result['jitter_pct']     = jitter_pct

    # ── Stage 6: Final decision ────────────────────────────────────────────────
    freq_ok     = blink_min_hz <= detected_hz <= blink_max_hz
    strength_ok = ac_strength >= autocorr_min_peak
    cycles_ok   = n_cycles >= min_cycles
    
    jitter_ok = True
    if n_pulses >= 2 and result.get('jitter_pct') is not None:
        jitter_ok = result['jitter_pct'] <= max_jitter_pct

    is_blinking = freq_ok and strength_ok and cycles_ok and jitter_ok

    if not strength_ok:
        reason = (
            f"Weak autocorrelation: {ac_strength:.2f} "
            f"(need >= {autocorr_min_peak}) at lag {best_lag} fr"
        )
    elif not freq_ok:
        reason = (
            f"{detected_hz:.2f} Hz outside [{blink_min_hz}, {blink_max_hz}] Hz  "
            f"(autocorr peak at lag {best_lag} fr)"
        )
    elif not cycles_ok:
        reason = f"Only {n_cycles} visible cycle(s), need >= {min_cycles}"
    elif not jitter_ok:
        reason = (
            f"Irregular spacing: MAD={result['mad_sp']:.1f} fr "
            f"({result['jitter_pct']:.0f}% jitter > {max_jitter_pct}% limit)"
        )
    else:
        sp_info = ''
        if result['median_spacing'] is not None:
            sp_info = (
                f"  |  edge median={result['median_spacing']:.1f} fr, "
                f"MAD={result['mad_sp']:.1f} fr"
            )
        reason = (
            f"OK  {detected_hz:.2f} Hz  |  autocorr={ac_strength:.2f}  |  "
            f"{n_cycles} cycles  |  lag={best_lag} fr{sp_info}"
        )

    result['is_blinking'] = is_blinking
    result['reason']      = reason
    
    # ── Stage 7: Temporal Segmentation (Frame-level State) ─────────────────────
    frame_state = np.full(n_sig, 'OFF', dtype=object)
    
    if is_blinking and len(rising_edges) >= 2:
        max_gap = best_lag * 1.5
        episodes = []
        curr_start = 0
        for i in range(1, len(rising_edges)):
            if (rising_edges[i] - rising_edges[i-1]) > max_gap:
                episodes.append((curr_start, i-1))
                curr_start = i
        episodes.append((curr_start, len(rising_edges) - 1))
        
        for (start_i, end_i) in episodes:
            if end_i >= start_i:
                start_frame = rising_edges[start_i]
                fe = falling_edges[falling_edges > rising_edges[end_i]]
                end_frame = fe[0] if len(fe) > 0 else (n_sig - 1)
                frame_state[start_frame:end_frame+1] = 'BLINKING'

    result['frame_state'] = frame_state

    return result

**Function Overview: `detect_blinks_per_camera`**
This function applies the core 1D blink detection algorithm iteratively to every tracked vehicle across camera views.

*Inputs:*

* **`df:`** The Pandas DataFrame containing geometric mask data generated by the upstream SAM 3 pipeline.
* **`fps:`** Camera framerate.
* **`detrend_window_sec, etc:`** Hyperparameters forwarded directly to `detect_blink_signal`.

*Core Mechanism:*

* **`Grouping:`** It groups the input dataframe by unique `track_id` to isolate the time-series history of individual vehicles.
* **`Signal Extraction:`** It iterates through the rows and safely parses the nested `amber_pct` metric from stringified dictionaries for all four possible indicator corners (FL, FR, RL, RR).
* **`Signal Processing:`** It invokes the core `detect_blink_signal` algorithm on each extracted time series.
* **`State Injection:`** It patches the resulting track-level `blink_state` and frame-level `light_on` boolean back into the original dataframe dictionaries.

*Outputs:*

* A modified Pandas DataFrame containing four new `{indicator}_blink_state` categorical columns and updated dictionary payloads.

In [ ]:
def detect_blinks_per_camera(
    df,
    fps,
    detrend_window_sec=2.5,
    detrend_noise_factor=0.3,
    min_detrended_range=5.0,
    autocorr_min_peak=0.3,
    min_on_frames=2,
    min_off_frames=2,
    blink_min_hz=0.8,
    blink_max_hz=2.5,
    min_cycles=2,
    max_jitter_pct=20.0,
):
    """
    detect_blinks_per_camera — camera-level blink detection using
    Detrend + Autocorrelation. Input is the output DataFrame of
    process_all_valid_images_with_sam (one row = one camera frame).

    Algorithm per (camera, track_id, indicator):
      Stage 1  Rolling-median baseline subtraction (removes slow drift).
      Stage 2  Flatness gate on the detrended signal.
      Stage 3  Autocorrelation — shape-agnostic periodic frequency detector.
      Stage 4  Binarize detrended signal for per-frame light_on state.
      Stage 5  Edge spacing (secondary confirmation).
      Stage 6  Final decision: freq_ok AND autocorr_strength_ok AND cycles_ok.

    Output per track:
      {col}_blink_state  in {'BLINKING', 'OFF'}

    light_on    -- True only when track is BLINKING and this frame is in the ON phase.
                   Always False when blink_state='OFF', even if amber_pct is nonzero.
    blink_state -- the track-segment verdict: BLINKING or OFF only.

    All original columns are preserved unchanged.

    Parameters
    ----------
    df                   : pd.DataFrame  — output of process_all_valid_images_with_sam
    fps                  : float         — video frame-rate (e.g. 10.0)
    detrend_window_sec   : rolling-median window in seconds (must be > 1 blink period)
    detrend_noise_factor : binarize threshold = detrended_std x this (noise floor)
    min_detrended_range  : p95-p5 of detrended must exceed this to pass flatness gate
    autocorr_min_peak    : min autocorrelation peak height to declare periodic signal
    min_on_frames        : short ON bursts below this are removed (noise)
    min_off_frames       : short OFF gaps below this are filled (waveform notches)
    blink_min_hz         : lower bound of valid blink frequency
    blink_max_hz         : upper bound of valid blink frequency
    min_cycles           : minimum complete ON->OFF cycles to declare BLINKING

    Returns
    -------
    pd.DataFrame — copy of df with:
        4 new *_blink_state columns (track-level verdict)
        light_on and blink_state fields patched into each indicator dict
    """
    
    # ── CONSTANTS ──────────────────────────────────────────────────────────────

    INDICATOR_COLS = [
        'rear_left_indicator',
        'rear_right_indicator',
        'front_left_indicator',
        'front_right_indicator',
    ]

    # ── HELPER FUNCTIONS ────────────────────────────────────────────────────────
    def patch_dict(val, **kwargs):
        """Inject key=value pairs into an indicator dict-string. Returns updated string."""
        s = str(val).strip() if val is not None else ''
        if s in ('', 'None', 'nan'):
            return val
        try:
            d = ast.literal_eval(s)
            d.update(kwargs)
            return str(d)
        except Exception:
            return val
    
    def parse_amber_pct(val):
        """Safely extract amber_pct from an indicator dict-string. Returns 0.0 on failure."""
        if val is None:
            return 0.0
        try:
            if pd.isna(val):
                return 0.0
        except (TypeError, ValueError):
            pass
        s = str(val).strip()
        if s in ('', 'None', 'nan'):
            return 0.0
        try:
            return float(ast.literal_eval(s).get('amber_pct', 0.0))
        except Exception:
            return 0.0
    
    
    df = df.copy()

    # Pre-create blink state columns (preserves order for all 4 even if col missing)
    for col in INDICATOR_COLS:
        blink_col = f'{col}_blink_state'
        if blink_col not in df.columns:
            df[blink_col] = pd.NA

    for (cam, track_id), grp in df.groupby(['camera', 'track_id'], sort=False):
        grp_sorted = grp.sort_values('timestamp')
        idx_sorted = grp_sorted.index   # original df row indices, sorted by time

        for col in INDICATOR_COLS:
            if col not in df.columns:
                continue

            # Build amber_pct signal for this (camera, track, indicator)
            signal = np.array([
                parse_amber_pct(df.at[i, col]) for i in idx_sorted
            ], dtype=float)

            res = detect_blink_signal(
                signal, fps,
                detrend_window_sec, detrend_noise_factor,
                min_detrended_range,
                autocorr_min_peak,
                min_on_frames, min_off_frames,
                blink_min_hz, blink_max_hz,
                min_cycles,
                max_jitter_pct,
            )

            # ── Frame-level State ──────────────────────────────────────────────────
            # Apply dynamic frame-wise blink states computed in Stage 7
            frame_states = res['frame_state']
            
            # Set the column dynamically per-frame
            df.loc[idx_sorted, f'{col}_blink_state'] = frame_states

            # ── Per-frame update ───────────────────────────────────────────────────
            for frame_i, orig_idx in enumerate(idx_sorted):
                f_state = frame_states[frame_i]
                frame_light_on = bool(res['cleaned'][frame_i]) if f_state == 'BLINKING' else False
                
                df.at[orig_idx, col] = patch_dict(
                    df.at[orig_idx, col],
                    light_on=frame_light_on,
                    blink_state=f_state,
                )

    return df

In [ ]:
df_camera = detect_blinks_per_camera(
    df_final_results,          # output of process_all_valid_images_with_sam
    fps=10.0,
    # Detrending
    detrend_window_sec=2.5,
    detrend_noise_factor=0.3,
    # Flatness gate
    min_detrended_range=5.0,
    # Autocorrelation
    autocorr_min_peak=0.3,
    # Run-length cleaning
    min_on_frames=2,
    min_off_frames=2,
    # Frequency validation
    blink_min_hz=0.8,
    blink_max_hz=2.5,
    min_cycles=2,
    max_jitter_pct=40.0,
)

In [ ]:
pd.set_option('display.max_columns', None) 
display(df_camera[(df_camera['track_id'] == 190) & (df_camera['camera'] == 'front_right_camera')].head(10))

### 4.8 Multi-Camera Data Fusion

Because the ego-vehicle's 6-camera setup possesses overlapping fields of view, a single tracked vehicle may be detected simultaneously by multiple cameras. To overcome spatial redundancy, we collect the indicator states for each unique vehicle at every discrete timestamp and gather them into a single, unified global record. During this fusion process, we prioritize positive detections—if *any* camera definitively registers an indicator as 'BLINKING', that active state overrides 'OFF' or null observations from adjacent cameras.

**Function Overview: `fuse_to_global`**
This function fuses parallel, camera-specific blink detections into a single, unified global coordinate frame across time, resolving overlapping fields of view. Additionally, it also outputs a 3D physical coordinate of the indicators.

*Inputs:*

* **`df_camera:`** The Pandas DataFrame outputted by `detect_blinks_per_camera`.

*Core Mechanism:*

* **`Spatial Aggregation:`** It groups identical `track_id` rows across all available camera views aligned to the exact same timestamp.
* **`Priority Resolution:`** It fuses left (Front Left + Rear Left) and right (Front Right + Rear Right) indicators, prioritizing a 'BLINKING' state if detected by *any* camera over an 'OFF' state.
* **`Attribute Collapse:`** It flattens redundant 3D physical traits (e.g., LiDAR `x_3d`, `y_3d`, `object_type`) by extracting the first non-null observation across the merged cameras.

*Outputs:*

* A condensed, global Pandas DataFrame mapping `timestamp` and `track_id` strictly to a unified `left_indicator`, `right_indicator`, along with 3D physical coordinates.

In [ ]:
def fuse_to_global(df_camera):
    """
    Fuses camera-level rows (output of detect_blinks_per_camera) into one
    global row per (timestamp, track_id), collapsing all camera views.

    Fusion rules
    ------------
    left_indicator   : highest-priority state across rear_left + front_left cameras
    right_indicator  : highest-priority state across rear_right + front_right cameras
    Priority         : BLINKING > OFF
    LiDAR columns   : first non-null value (object-level, camera-independent)
    object_type      : mode (most frequent) across cameras for that timestamp+track

    Two datasets are produced by this pipeline:
      df_camera = detect_blinks_per_camera(...)  -> camera-level, all columns preserved
      df_global = fuse_to_global(df_camera)      -> timestamp-level, merged indicators

    Parameters
    ----------
    df_camera : pd.DataFrame — output of detect_blinks_per_camera

    Returns
    -------
    pd.DataFrame with columns:
        timestamp, track_id, object_type,
        left_indicator, right_indicator,
        [available LiDAR columns: x_3d, y_3d, z_3d, yaw_3d, width_3d, length_3d]
    """

    def highest_priority_state(states):
        """
        Returns the highest-priority state from a list.
        Priority: BLINKING > OFF
        """

        STATE_PRIORITY = {'BLINKING': 1, 'OFF': 0}
        best = 'OFF'
        for s in states:
            if isinstance(s, str) and STATE_PRIORITY.get(s, -1) > STATE_PRIORITY[best]:
                best = s
        return best

    # LiDAR / 3D object columns (camera-independent, taken from first non-null camera)
    LIDAR_COLS = ['x_3d', 'y_3d', 'z_3d', 'yaw_3d', 'width_3d', 'length_3d']
    
    rows = []

    for (ts, track_id), grp in df_camera.groupby(['timestamp', 'track_id'], sort=False):

        # ── Left / Right indicator fusion ─────────────────────────────────────
        def collect(cols):
            states = []
            for col in cols:
                blink_col = f'{col}_blink_state'
                if blink_col in grp.columns:
                    states += grp[blink_col].dropna().tolist()
            return states

        left_indicator  = highest_priority_state(
            collect(['rear_left_indicator', 'front_left_indicator'])
        )
        right_indicator = highest_priority_state(
            collect(['rear_right_indicator', 'front_right_indicator'])
        )

        # ── Object type ───────────────────────────────────────────────────────
        if 'object_type' in grp.columns:
            mode_vals   = grp['object_type'].dropna()
            object_type = mode_vals.mode().iloc[0] if len(mode_vals) else None
        else:
            object_type = None

        row = {
            'timestamp':       ts,
            'track_id':        track_id,
            'object_type':     object_type,
            'left_indicator':  left_indicator,
            'right_indicator': right_indicator,
        }

        # ── LiDAR / 3D columns (first non-null value across cameras) ──────────
        for lcol in LIDAR_COLS:
            if lcol in grp.columns:
                valid    = grp[lcol].dropna()
                row[lcol] = valid.iloc[0] if len(valid) else None

        rows.append(row)

    if not rows:
        return pd.DataFrame()

    global_df = pd.DataFrame(rows)

    # Build final column order: core first, then available LiDAR cols
    core_cols     = ['timestamp', 'track_id', 'object_type',
                     'left_indicator', 'right_indicator']
    lidar_present = [c for c in LIDAR_COLS if c in global_df.columns]

    return (
        global_df[core_cols + lidar_present]
        .sort_values(['timestamp', 'track_id'])
        .reset_index(drop=True)
    )

In [ ]:
df_global = fuse_to_global(df_camera)

In [ ]:
pd.set_option('display.max_columns', None) 
display(df_global[df_global['track_id'] == 190].head(20))

#### Visualisation of the result

*Only possible if dataset is copied already*

In [ ]:
from functions.visualise import show_pipeline_frame

show_pipeline_frame(
    timestamp  = df_global['timestamp'].iloc[422],
    df_camera  = df_camera,
    df_fused   = df_global,
    calib_data = calib_data,
    seq_dir    = seq_dir,
)

## 5 Experiments

The following experiments can be performed independent of the 'data_night_sample' dataset.

Numerous such experiments were conducted by us to understand the models, develop functions and tweak parameters as per requirement. Now its presented in an easy to understand format.

In [ ]:
calib_data = load_calibration_file(os.path.join('topic-17-files/', 'calibration.json'))

### 5.1 SAM 3.1 Parameter Tuning

Foundation models like SAM 3.1 rely heavily on hyperparameter constraints to differentiate between true targets and background noise. This interactive experiment isolates two critical control variables: the Object-Level Confidence Threshold (`conf_threshold`) and the Pixel-Level Mask Threshold (`mask_threshold`). By systematically using these values in an example image, we can visually evaluate the model's sensitivity to vehicle indicators. This visual study allows us to empirically determine the optimal parameter values that maximizes masking effeciency while minimizing false positives caused by environmental lighting anomalies.

**Expected Results & Trade-Offs:**
*   **Low Thresholds (e.g., Conf: 0.2, Mask: 0.2) - Over-segmentation:** The model becomes sensitive, including excessive pixels. It wraps not just the physical indicator light, but also the surrounding optical glare, reflections on the vehicle's paint, and potentially unrelated bright spots.
*   **High Thresholds (e.g., Conf: 0.8, Mask: 0.8) - Under-segmentation:** The model becomes conservative. It rejects noise but also entirely miss faint indicators, off-angle lights, or shrink the mask that it only captures the brightest spots of the indicator rather than the actual shape of the lens cover.
*   **The Optimal Sweet Spot:** At the balanced intersection (typically around Conf: 0.4–0.6 and Mask: 0.5–0.7), the masks "shrink-wrap" the physical boundaries of the indicator covers.

#### 5.1.1 Prompt
*   **Direct indicator extraction:** It was found that SAM3 is incapable of extracting only the indicators but on the other had is capable of picking the whole head- or tail- light cluster. This can be checked by changing the prompt to "indicators" and contrasting it against "taillights"
*   **Length of prompt:** Its evident that having long prompts like "extract the taillight indicators of cars only", did not show any improvement as compared to "car taillights". Even the shorter prompt tends to be mis-obeyed in several cases and it so happens that this prompt even picks scooter and truck lamp clusters too.

In [ ]:
from functions.experiments import run_single_sam3_inference

run_single_sam3_inference(
    image_path          = "topic-17-files/segmentation/indicator_segmentation_fr_1.jpg",
    cam_name            = "front_right_camera",
    calib_data          = calib_data,
    generate_sam3_masks = generate_sam3_masks, 
    processor           = processor,
    model               = model,
    device              = device,
    text_prompt         = "car taillight",
    exclude_prompt      = None,
    conf_threshold      = 0.50,
    mask_threshold      = 0.60
)

#### 5.1.2 Effect of Varying Mask Threshold
When the confidence score is started off from deliberately low values and then increased gradually, we observe that the overall area of the SAM 3 masks decreases as the mask threshold increases. Lower mask thresholds capture the hazy "bloom" surrounding active lights, while higher thresholds shrink the mask down to a smaller area within the whole true lamp area.

In [ ]:
from functions.experiments import run_mask_threshold_ablation

run_mask_threshold_ablation(
    image_path          = "topic-17-files/segmentation/indicator_segmentation_fr_2.jpg",
    cam_name            = "front_right_camera",
    calib_data          = calib_data,
    generate_sam3_masks = generate_sam3_masks,
    processor           = processor,
    model               = model,
    device              = device,
    text_prompt         = "car taillight",
    fixed_conf          = 0.30
)

#### 5.1.3 Effect of Varying Confidence Score
At lower confidence scores, the model struggles frequent misclassification due to bright lighting (from oncoming headlights or streelights) falling on the ego during the night or environmental overexposure (sunlight or reflections hitting the lamps). As the confidence score is increased, these false-positive object classifications are reduced, ensuring the model only segments valid turn indicator clusters.

In [ ]:
from functions.experiments import run_conf_threshold_ablation

run_conf_threshold_ablation(
    image_path          = "topic-17-files/segmentation/indicator_segmentation_fr_1.jpg",
    cam_name            = "front_right_camera",
    calib_data          = calib_data,
    generate_sam3_masks = generate_sam3_masks, 
    processor           = processor,
    model               = model,
    device              = device,
    text_prompt         = "car taillight",
    fixed_mask          = 0.60
)

Based on these observations, the final SAM 3.1 parameters were set as follows:

| Parameter Name | Final Configuration |
| :--- | :--- |
| `text_prompt` | "car headlight", "car taillight" |
| `sam_conf_threshold` | 0.50 |
| `sam_mask_threshold` | 0.60 |
| `min_mask_area` | 25 pixels |
---

### 5.2 Depth Map Fusion using LiDAR Data

To extract accurate 3D coordinates from the 2D camera masks, we require an absolute, metric depth map. Since Depth Anything 3's (DA3) absolute model is inaccurate we use DA3-Gaint that outputs relative depth. Hence, we must then scale it using the sparse ground-truth points provided by the LiDAR. Different fusion scaling methods were analyzed:
*   **Global Median Scaling:** Computes a single scalar multiplier by taking the median ratio of LiDAR depth to DA3 depth across the entire frame. While computationally simple, it fails to account for perspective distortion and non-linear depth compression at the edges of the image.
*   **Zonal / Localized Scaling (Chosen Method):** Instead of calculating a global scale factor, this method computes localized scaling multipliers for every valid LiDAR point mapped to a DA3 pixel.

In [ ]:
from functions.experiments import compare_raw_depth_vs_lidar

compare_raw_depth_vs_lidar(
    image_name  = "1750163922950978994.jpg",
    cam_name    = "front_right_camera",
    df_cam      = pd.read_csv('topic-17-files/depth_map/data.csv'),
    calib_data  = calib_data,
    depth_model = model_DA3,   
    device      = device,
    seq_dir     = "topic-17-files/depth_map"
)

### 5.3 Mask Association

Post mask generation, the pipeline must accurately associate each indicator mask to a specific vehicle track ID in the dataset. While our 3D geometric association logic is highly precise, experiments reveal that at far distances (>30 meters), the 3D association begins to become inaccurate. This occurs due to two factors: the extreme sparsity of LiDAR points striking distant vehicles, and increasing noise in DA3's depth estimation at long ranges. To maintain pipeline integrity, the system seamlessly falls back to 2D Bounding Box association when 3D validation fails, ensuring distant vehicles are still accurately monitored for blinking indicators.

In [ ]:
from functions.experiments import visualize_final_dataframe_per_mask

visualize_final_dataframe_per_mask(
    df_frame   = pd.read_csv('topic-17-files/association/ambiguous-2d/1750165030811108112_results.csv'), 
    image_path = 'topic-17-files/association/ambiguous-2d/back_right_camera/1750165030811108112.jpg', 
    calib_data = calib_data
)

### 5.4 Amber Extraction (CIELAB)

Visualizations of the amber pixel detections were conducted to evaluate the robustness of the CIELAB color space.

#### 5.4.1 Effect of Varying $r_{max}$ (Maximum Distance)
The $r_{max}$ parameter controls the radius of the acceptance sphere centered around the set ideal amber coordinate in 3D CIELAB space. Increasing $r_{max}$ makes the detector more lenient, accepting pale yellows and darker oranges. However, setting $r_{max}$ too high causes the sphere to intersect with white light regions (headlight glare) or red light regions (brake lights), drastically increasing false positives.

In [ ]:
from functions.experiments import run_max_dist_sweep

run_max_dist_sweep(
    image_path      = "topic-17-files/color_filtering/distance_sweep/1750164055850975037.jpg",
    mask_path       = "topic-17-files/color_filtering/distance_sweep/1750164055850975037.npz",
    mask_idx        = 1,
    cam_name        = "front_right_camera",
    calib_data      = calib_data,
    fixed_b_min     = 135,
    max_dist_values = [50, 80, 120],
)

#### 5.4.2 Effect of Varying $b_{min}$ (Yellowness Floor)
The $b_{min}$ parameter serves as a hard boundary on the Blue-Yellow axis. At night, extreme luminance from red brake lights and white headlights can cause their perceived CIELAB coordinates to drift closer to the defined amber sphere. Increasing $b_{min}$ forces the system to reject any pixel that does not possess a strong, yellow tint, effectively slicing off the bottom of the acceptance sphere to eliminate false positives from intense red/white bleed-over. This explains why the optimized nighttime $b_{min}$ thresholds are significantly stricter (170-180) than the daytime thresholds (135-155).

In [ ]:
from functions.experiments import run_bmin_sweep

run_bmin_sweep(
    image_path     = "topic-17-files/color_filtering/bmin_sweep/1750884698150979996.jpg",
    mask_path      = "topic-17-files/color_filtering/bmin_sweep/1750884698150979996.npz",
    mask_idx       = 3,
    cam_name       = "front_right_camera",
    calib_data     = calib_data,
    fixed_max_dist = 80,
    b_min_values   = [0, 150, 190],
)

Based on these specific lighting challenges, the final photometric thresholds were distinctly tuned for daytime and nighttime conditions:

| Parameter Category | Parameter Name | Day Configuration | Night Configuration |
| :--- | :--- | :--- | :--- |
| **Photometric (Rear)** | `rear_max_dist` | 80 | 80 |
| | `rear_b_min` | 135 | 170 |
| | `rear_expansion` | 0.20 | 0.20 |
| **Photometric (Front)** | `front_max_dist` | 80 | 80 |
| | `front_b_min` | 155 | 180 |
| | `front_expansion` | 0.20 | 1.00 |

---

### 5.5 Blink Signal Processing

The 1D blink detection algorithm here is subjected to hyperparameter variation to balance frequency based sensitivity against environmental noise rejection.

#### 5.5.1 Changing Autocorrelation Minimum Peaks
The `autocorr_min_peak` dictates the minimum required strength of the periodic signal. Lowering this value allows the system to detect faint, irregular, or partially occluded blinks. However, if set too low, the system will incorrectly flag random environmental noise fluctuations or sensor flickering as a valid turn signal.

In [ ]:
from functions.experiments import run_autocorr_sweep

run_autocorr_sweep(
    raw_csv_path  = "topic-17-files/frequency_filtering/blinking_sample_1.csv",
    gt_csv_path   = "topic-17-files/frequency_filtering/blinking_sample_1_groundtruth.csv",     
    track_id      = 941,
    cam_name      = "front_left_camera",
    indicator_col = "rear_right_indicator",
    fps           = 10.0,
    autocorr_values      = (0.15, 0.30, 0.50),
    fixed_min_range      = 5.0,
    fixed_detrend_window = 2.5,
)

#### 5.5.2 Changing Detrend Window Values
The `detrend_window_sec` dictates the width of the rolling median filter window used to adapt to the environmental exposure. 
*   **Window too small:** The median filter tracks the signal too closely, actively subtracting the blink pulses themselves and flattening the valid signal to zero.
*   **Window too large:** The filter responds too slowly, failing to track sudden environmental shadows or the rapid approach of oncoming headlights, causing the detrended signal to drift wildly outside the binarization thresholds.

In [ ]:
from functions.experiments import run_detrend_window_sweep

run_detrend_window_sweep(
    raw_csv_path  = "topic-17-files/frequency_filtering/blinking_sample_2.csv",
    gt_csv_path   = "topic-17-files/frequency_filtering/blinking_sample_2_groundtruth.csv",     
    track_id      = 1832,
    cam_name      = "front_right_camera",
    indicator_col = "rear_right_indicator",
    fps           = 10.0,
    window_values  = (0.1, 0.6, 2.5),
    fixed_autocorr = 0.30,
    fixed_min_range = 5.0,
)

Following these sweeps, the final signal processing parameters were configured as:

| Parameter Category | Parameter Name | Final Configuration |
| :--- | :--- | :--- |
| **Signal Detrending** | `detrend_window_sec` | 2.5 sec |
| | `detrend_noise_factor`| 0.3 |
| | `min_detrended_range` | 5.0 % |
| **Autocorrelation** | `autocorr_min_peak` | 0.3 |
| **Temporal Cleaning** | `min_on_frames` | 2 frames |
| | `min_off_frames` | 2 frames |
| **Frequency Validation**| `blink_min_hz` | 0.8 Hz |
| | `blink_max_hz` | 2.5 Hz |
| | `min_cycles` | 2 |
| | `max_jitter_pct` | 40.0 % |
| **Camera Settings** | `fps` | 10.0 |

---

## 6 Evaluation

More details about evaluation tool is documented in Evaluation folder along with original code and datasets.

### 6.1 Evaluation Dataset

The pipeline was evaluated manually using a **stratified random sample** drawn from two sequences:
a daytime drive and a nighttime drive recorded with the same sensor suite.

#### 6.1.1 How frames were selected

Evaluating all frames from the full recording is not feasible.
Hence, a balanced subset was sampled using the following strategy:

- **50 % of frames** were selected from frames where the pipeline predicted at least one indicator as **ON**
- **50 % of frames** were selected from frames where the pipeline predicted all indicators as **OFF**

This balance prevents the metrics from being biased.

Within each pool, frames were distributed **evenly across all cameras**
(front-left, front-right, rear-left, rear-right) to avoid any single viewpoint skewing.

#### 6.1.2 Evaluation split

| Sequence | Frames Evaluated | Sampling file |
|---|---|---|
| **Day** | 85 | `Input/sam3_evaluation_sample_day_part_1.csv` + `_part_2.csv` |
| **Night** | 36 | `Input/sam3_evaluation_sample_night_part_1.csv` |
| **Total** | **121** | `Output/results_sam3_evaluation_day.csv` / `_night.csv` |

> **Note:** The night sample is smaller because fewer frames passed the quality filter
> (frames with ≥ 3 tracked vehicles and all mask data available in csv).

#### What each chart shows

The four pipeline stages are evaluated — an error at an earlier stage
stops the evaluation of later stages for that detection. The queries we as follows:

1. **SAM3 Segmentation** — Did the model produce a mask on a real indicator?
2. **Association** — Was that mask assigned to the correct vehicle?
3. **Classification** — Was the indicator side (rear-left, front-right, …) correct?
4. **Light ON/OFF** — Did the predicted state (blinking / steady / off) match reality?

> The charts below show the **raw indicator counts** (not frame-level accuracy),
> since one frame can contain multiple indicators.


In [ ]:
CSV_1  = 'evaluation/Output/results_sam3_evaluation_day.csv'
CSV_2  = 'evaluation/Output/results_sam3_evaluation_night.csv'
LABEL_1 = 'Day'
LABEL_2 = 'Night'

sums1 = pd.read_csv(CSV_1).sum(numeric_only=True)
sums2 = pd.read_csv(CSV_2).sum(numeric_only=True)

COLOR_MAP = {'Correct': '#2ecc71', 'Missed':  '#f39c12', 'Wrong':   '#e74c3c',}

def _pie(ax, values, labels, title):
    """Draw one pie, skip zero-value slices."""
    pairs  = [(v, l) for v, l in zip(values, labels) if v > 0]
    if not pairs:
        ax.text(0.5, 0.5, 'No Data', ha='center', va='center', fontsize=14, weight='bold')
        ax.set_title(title, fontsize=14, weight='bold'); ax.axis('off'); return
    vals, labs = zip(*pairs)
    colors = []
    for l in labs:
        colors.append(next((c for k, c in COLOR_MAP.items() if l.startswith(k)), '#aaaaaa'))
    total = sum(vals)
    ax.pie(vals, labels=labs, colors=colors,
           autopct=lambda p: f"{int(round(p * total / 100))}",
           textprops={'fontsize': 13, 'weight': 'bold'},
           startangle=90, wedgeprops={'edgecolor': 'black', 'linewidth': 1.2})
    ax.set_title(title, fontsize=14, weight='bold')

fig1, axes = plt.subplots(2, 3, figsize=(20, 12), facecolor='white')
fig1.suptitle('Pipeline Accuracy — Pie Chart Comparison', fontsize=22, weight='bold', y=1.01)

for row_i, (sums, label) in enumerate([(sums1, LABEL_1), (sums2, LABEL_2)]):
    # Left row label
    axes[row_i, 0].annotate(label, xy=(-0.35, 0.5), xycoords='axes fraction', fontsize=18, weight='bold', color='#333333', va='center', ha='right', rotation=90)
    _pie(axes[row_i, 0],[sums['sam3_tp'], sums['sam3_fn'], sums['sam3_fp']],['Correct (TP)', 'Missed Masks (FN)', 'Wrong (FP)'],'1. SAM3 Segmentation')
    _pie(axes[row_i, 1],[sums['assoc_tp'], sums['assoc_fp']],['Correct (TP)', 'Wrong (FP)'],'2. Association')
    _pie(axes[row_i, 2],[sums['class_tp'], sums['class_fp']],['Correct (TP)', 'Wrong (FP)'],'3. Classification')

# Divider line between rows
line = plt.Line2D([0.05, 0.95], [0.5, 0.5], transform=fig1.transFigure, color='#cccccc', linewidth=1.5, linestyle='--')
fig1.add_artist(line)
plt.tight_layout()
plt.show()

light_labels = ['True Positive', 'False Negative', 'True Negative', 'False Positive']
light_keys   = ['light_tp', 'light_fn', 'light_tn', 'light_fp']
bar_colors   = ['#2ecc71', '#c0392b', '#27ae60', '#e67e22']

counts1 = [sums1[k] for k in light_keys]
counts2 = [sums2[k] for k in light_keys]

x     = np.arange(len(light_labels))
width = 0.35

fig2, ax = plt.subplots(figsize=(14, 7), facecolor='white')
ax.set_facecolor('white')

bars1 = ax.bar(x - width/2, counts1, width, label=LABEL_1, color=bar_colors, edgecolor='black', linewidth=1.2, alpha=0.95)
bars2 = ax.bar(x + width/2, counts2, width, label=LABEL_2, color=bar_colors, edgecolor='black', linewidth=1.2, alpha=0.55, hatch='//')

ax.set_title('4. Light ON/OFF Accuracy — Comparison', fontsize=20, weight='bold', pad=16)
ax.set_ylabel('Number of Indicators', fontsize=14, weight='bold')
ax.set_xticks(x); ax.set_xticklabels(light_labels, fontsize=13)
ax.tick_params(axis='y', labelsize=12)
ax.grid(axis='y', linestyle='--', color='#dddddd', alpha=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        if h > 0: ax.text(bar.get_x() + bar.get_width()/2, h + max(max(counts1), max(counts2)) * 0.01, f'{int(h)}', ha='center', va='bottom', fontsize=12, weight='bold')

# Custom legend (solid = CSV1, hatched = CSV2)
import matplotlib.patches as mpatches
leg_handles = [mpatches.Patch(facecolor='#888888', edgecolor='black', label=LABEL_1), mpatches.Patch(facecolor='#888888', edgecolor='black', hatch='//', label=LABEL_2),]
ax.legend(handles=leg_handles, fontsize=13, framealpha=0.9)

plt.tight_layout()
plt.show()

#### 6.1.3 Depth Based Association Analysis

A visualization of quantity of masks associated using 3D Dense depth v/s 2D bounding box method. This is obtained to judge the ideal working range for the models used and effectiveness of association using 3D. It can be seen that 30 meters is the sweet spot for good depth estimation and detection.

In [ ]:
STATS_CSV = 'evaluation/depth_association_stats.csv'


BG_COLOR = '#FFFFFF'
plt.style.use('default')
plt.rcParams.update({'figure.facecolor': BG_COLOR,'axes.facecolor': BG_COLOR,'savefig.facecolor': BG_COLOR,'text.color': '#333333','axes.labelcolor': '#333333','xtick.color': '#333333','ytick.color': '#333333'})

if not os.path.exists(STATS_CSV):
    print(f"Error: {STATS_CSV} not found. Run the processing cell first.")
else:
    master_df = pd.read_csv(STATS_CSV)
    datasets = master_df['dataset'].unique()
    
    for dataset in datasets:
        # Isolate data for this dataset
        agg_df = master_df[master_df['dataset'] == dataset].copy()
        
        # Calculate success rates safely
        agg_df['total_masks'] = agg_df['count_3d_success'] + agg_df['count_2d_fallback']
        agg_df['success_rate_3d'] = (agg_df['count_3d_success'] / agg_df['total_masks']) * 100
        agg_df['success_rate_3d'] = agg_df['success_rate_3d'].fillna(0)  
        
        # Setup plot
        fig, ax1 = plt.subplots(figsize=(12, 6))
        x = np.arange(len(agg_df))
        bar_width = 0.4
        
        # Bar plots
        ax1.bar(x - bar_width/2, agg_df['count_3d_success'], width=bar_width, color='green', label='3D Association (Tag 1)')
        ax1.bar(x + bar_width/2, agg_df['count_2d_fallback'], width=bar_width, color='orange', label='2D Fallback (Tag 0)')
        
        ax1.set_xlabel('Depth Bin (meters)', fontsize=12)
        ax1.set_ylabel('Number of Masks', fontsize=12)
        ax1.set_title(f'Mask Association Method by Depth - {dataset}', fontsize=14, fontweight='bold')
        ax1.set_xticks(x)
        ax1.set_xticklabels(agg_df['depth_bin_str'], rotation=45, ha='right')
        ax1.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Line plot for 3D success rate on secondary y-axis
        ax2 = ax1.twinx()
        ax2.plot(x, agg_df['success_rate_3d'], color='blue', marker='o', linewidth=2, label='3D Success Rate %')
        ax2.set_ylabel('Depth Map Success Rate (%)', color='blue', fontsize=12)
        ax2.tick_params(axis='y', labelcolor='blue')
        ax2.set_ylim(0, 105)
        
        # Combine legends from both axes
        lines_1, labels_1 = ax1.get_legend_handles_labels()
        lines_2, labels_2 = ax2.get_legend_handles_labels()
        ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right')
        
        plt.tight_layout()
        plt.show()

## 7 Conclusion

We presented a fully automated pipeline using the DrivIng dataset for auto-labeling turn indicator states. This pipeline is split between four stages with the first being segmentation of turn indicators using SAM3 foundation model. It is evident from our study that SAM3 is incapable of extracting only turn indicators directly, but on the other hand was capable of extracting the head-and tail- light clusters from camera frames. In order to associate the SAM3 detections to the right track IDs we required an accurate dense depth map, which was achieved by fusing relative depth obtained from DA3 another foundation model, with already available LiDAR data from the dataset. This fusion was critical to overcome the sparse nature of LiDAR. The third stage involves geometric association in which we project 2D indicator mask centroids into 3D space using the fused depth information which allows us to associate the masks to the right bounding boxes. Thereafter, we apply a classification function to rightly tag the indicators as left or right. Finally in the fourth stage, since we encountered challenges using only colour logic due to reasons such as but not limited to overexposure in dynamic lighting conditions and faded lamp clusters; we took advantage of the frequency nature of the indicator and employed a dual strategy – colour based and frequency based – to estimate the turn  indicator state. Hence, we first use CIELAB color space – due to its ease of tunability – to extract the Amber coloured blinker pixels first. Subsequently, we apply frequency based filtering which consists of rolling median, alongside autocorrelation to overcome shortfalls of colour only methods. It was concluded that this algorithm was very effective between varying lighting conditions across day and night data, whilst it could be improved further with fine tuning of the amber colour extraction.

From our study it is evident that the automatic annotation of indicators is possible, but is bottlenecked primarily by SAM3 extractions. It was found that SAM3 was inconsistent with its detections - in several cases, during the day it completely skipped masking the turn indicators due to split cluster setup in certain car models requiring vertical mask elongation for compensation, or other cases where only partial areas of lamps were masked due to background lighting; Furthermore, due to bright headlights bleeding and essentially masking the indicators especially during the night. Overall it was found that our model worked effectively within 30 meters range even in dense traffic situations. 

## 8 Future Scope

As future scope SAM3 needs to be studied more in depth and the prompt to be tuned accordingly. For instance the current pipeline requests the segmentation model to scan each image twice — headlights and taillights — which doubles the processing time. A recently proposed study called DART [[13]](#ref) offers a smarter approach: it scans the image only once and searches for both types of lights at the same time, making it theoretically twice as fast.. Adopting this strategy in future work could significantly reduce computation and time required. Furthermore, The current dataset lacks RAW exposure data from cameras, which could prove useful to overcome natural and artificial lighting issues, this could benefit both the foundation models, and the Indicator state estimation algorithm. To strengthen the current methodology and to overcome shortfalls of SAM3, maybe training a neural network on the TLD: A Vehicle Tail Light signal Dataset and Benchmark [[14]](#ref) could be explored to complement SAM3 detections.Reduction in processing could be achieved by eliminating parked vehicles from analysing possibly by using map data.

## 9 <a id="ref"></a> References

*The list of references lists further information about all work that has been cited. Please make sure to hyperlink all references.*

| # | Year | Authors | Title |
| --- | --- | --- | --- |
| 1 | 2026 | Dominik Rößle, Xujun Xie, Adithya Mohan, Venkatesh Thirugnana Sambandham, Daniel Cremers, Torsten Schön | [DrivIng: A Large-Scale Multimodal Driving Dataset with Full Digital Twin Integration](https://doi.org/10.48550/arXiv.2601.15260) |
| 2 | 2026 | Meta Fundamental AI Research (FAIR) | [Segment Anything 3, Hugging Face Repository](https://huggingface.co/facebook/sam3) |
| 3 | 2026 | Haotong Lin, Sili Chen, Junhao Liew, Donny Y. Chen, Zhenyu Li, Guang Shi, Jiashi Feng, Bingyi Kang | [Depth Anything 3: Recovering the Visual Space from Any Views](https://arxiv.org/abs/2511.10647) |
| 4 | 2019 | Holger Caesar, Varun Bankiti, Alex H. Lang, Sourabh Vora, Venice Erin Liong, Qiang Xu, Anush Krishnan, Yu Pan, Giancarlo Baldan, Oscar Beijbom | [nuScenes: A multimodal dataset for autonomous driving](https://doi.org/10.48550/arXiv.1903.11027)|
| 5 | 2025 | Runsheng Xu, Hubert Lin, Wonseok Jeon, Hao Feng, Yuliang Zou, Liting Sun, John Gorman, Ekaterina Tolstaya, Sarah Tang, Brandyn White, Ben Sapp, Mingxing Tan, Jyh-Jing Hwang, Dragomir Anguelov | [WOD-E2E: Waymo Open Dataset for End-to-End Driving in Challenging Long-tail Scenarios](https://doi.org/10.48550/arXiv.2510.26125)|
| 6 | 2021 | Yiyi Liao, Jun Xie, Andreas Geiger | [KITTI-360: A Novel Dataset and Benchmarks for Urban Scene Understanding in 2D and 3D](https://doi.org/10.48550/arXiv.2109.13410)|
| 7 | 2024 | ECE UN Regulation | [E/ECE/324/Rev.1/Add.47/Rev.13−E/ECE/TRANS/505/Rev.1/Add.47/Rev.13]()|
| 8 | 2020 | ECE UN Regulation | [E/ECE/TRANS/505/Rev.3/Add.147]()|
| 9 | 2025 | Till Beemelmanns | [Depth Anything V3 TensorRT ROS 2 Node](https://github.com/ika-rwth-aachen/ros2-depth-anything-v3-trt)|
| 10 | v2026 | RWTH Aachen University | [Automated and Connected Driving Challenges](https://www.edx.org/learn/automation/rwth-aachen-university-automated-and-connected-driving-challenges)|
| 11 | 2012 | Mauricio Casares, Akhan Almagambetov, Senem Velipasalar | [A Robust Algorithm for the Detection of Vehicle Turn Signals and Brake Lights](https://doi.org/10.1109/AVSS.2012.2)|
| 12 | 2017 | Han-Kai Hsu1 , Yi-Hsuan Tsai, Xue Mei, Kuan-Hui Lee, Naoki Nagasaka, Danil Prokhorov, Ming-Hsuan Yang | [Learning to Tell Brake and Turn Signals in Videos Using CNN-LSTM Structure](https://doi.org/10.1109/ITSC.2017.8317782)|
| 13 | 2026 | Mehmet Kerem Turkcan | [Detect Anything in Real Time: From Single-Prompt Segmentation to Multi-Class Detection](https://arxiv.org/abs/2603.11441) |
| 14 | 2024 | Jinhao Chai, Shiyi Mu, Shugong Xu | [TLD: A Vehicle Tail Light signal Dataset and Benchmark](https://doi.org/10.48550/arXiv.2409.02508)|